# EfficientZero V2 на шахматах

Самостоятельный ноутбук: алгоритм, поиск, реплей и шахматная среда лежат в ячейках ниже. Пакет reinforcement-studio не импортируется.

Нужны `torch`, `gymnasium`, `numpy` и `open_spiel` (`import pyspiel`). OpenSpiel считает правила шахмат: 4674 действия, наблюдение `20×8×8`, легальные ходы и мат/пат/повторы.

Что внутри, без сокращения алгоритма:

- представление, динамика с LSTM value-prefix и головы policy/value;
- категориальные value и reward на гиперболической шкале;
- SimSiam consistency loss с BatchNorm и stop-gradient;
- Gumbel search с Sequential Halving, в том числе маска легальных ходов в корне и не больше 32 детей во внутренних узлах;
- self-play за обе стороны, бэкап с дисконтом `-γ`, материальное промежуточное вознаграждение;
- приоритетный реплей. Reanalyze для двух игроков выключен в исходном коде: без сохранённой маски легальных ходов он портит цель политики.


In [ ]:
import math
import os
from pathlib import Path
from typing import Any, Sequence

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from gymnasium import spaces
from gymnasium.vector import VectorEnv

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
print(f"device={DEVICE}  torch={torch.__version__}")


In [ ]:
"""Шахматы: движок OpenSpiel и Gym-обёртка self-play.

Правила, легальные ходы и терминальные исходы считает OpenSpiel (`pyspiel`).
Наблюдение — тензор 8×8×20 с точки зрения стороны, которая ходит. Награда
за обычный ход — нормированная разница материала хода сделавшего ход,
в конце партии +1/0/−1. Лимит 512 полуходов, как в исходной обёртке.
"""
from abc import ABC, abstractmethod

def count_in_direction(board: np.ndarray, row: int, col: int, dr: int, dc: int, player: int) -> int:
    rows, cols = board.shape
    count = 0
    r, c = row + dr, col + dc
    while 0 <= r < rows and 0 <= c < cols and board[r, c] == player:
        count += 1
        r += dr
        c += dc
    return count


def check_win_at(board: np.ndarray, row: int, col: int, player: int, k: int) -> bool:
    """True if placing `player` at (row, col) completes a run of length k."""
    for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
        total = 1 + count_in_direction(board, row, col, dr, dc, player) + count_in_direction(
            board, row, col, -dr, -dc, player
        )
        if total >= k:
            return True
    return False


class BoardGame(ABC):
    id: str
    name: str
    rows: int
    cols: int
    win_length: int

    def __init__(self) -> None:
        self.board = np.zeros((self.rows, self.cols), dtype=np.int8)
        self.current_player = 1
        self.done = False
        self.winner: int | None = None  # +1, -1, or 0 for draw
        self.last_move: tuple[int, int] | None = None

    @property
    def action_size(self) -> int:
        return self.rows * self.cols

    @property
    def input_planes(self) -> int:
        """Channels in `encode()`. Built-in stone games use 3; OpenSpiel chess is 20."""
        return 3

    def reset(self) -> np.ndarray:
        self.board = np.zeros((self.rows, self.cols), dtype=np.int8)
        self.current_player = 1
        self.done = False
        self.winner = None
        self.last_move = None
        return self.encode()

    @abstractmethod
    def legal_actions(self) -> list[int]:
        ...

    @abstractmethod
    def step(self, action: int) -> tuple[np.ndarray, bool, int | None]:
        """Applies `action` for the current player. Returns (encoded_state, done, winner)."""
        ...

    def encode(self) -> np.ndarray:
        """3 planes from the perspective of the player to move: own stones,
        opponent stones, and a constant plane (helps the conv net know whose
        turn it is when the board itself is symmetric, e.g. an empty board)."""
        own = (self.board == self.current_player).astype(np.float32)
        opp = (self.board == -self.current_player).astype(np.float32)
        turn = np.full_like(own, 1.0 if self.current_player == 1 else 0.0)
        return np.stack([own, opp, turn], axis=0)

    def clone(self) -> "BoardGame":
        clone = self.__class__()
        clone.board = self.board.copy()
        clone.current_player = self.current_player
        clone.done = self.done
        clone.winner = self.winner
        clone.last_move = self.last_move
        return clone

    def render_ascii(self, symbols: tuple[str, str, str] = (".", "X", "O")) -> str:
        chars = {0: symbols[0], 1: symbols[1], -1: symbols[2]}
        return "\n".join(
            " ".join(chars[int(v)] for v in row) for row in self.board
        )

    def board_list(self) -> list[list[int]]:
        return self.board.tolist()

import threading
from typing import Any

import numpy as np


_GAME_CACHE: dict[str, Any] = {}

_FEN_PIECE = {
    "P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6,
    "p": -1, "n": -2, "b": -3, "r": -4, "q": -5, "k": -6,
}


def _pyspiel_game(name: str):
    game = _GAME_CACHE.get(name)
    if game is None:
        import pyspiel

        game = pyspiel.load_game(name)
        _GAME_CACHE[name] = game
    return game


_OBSERVERS = threading.local()


def _observer(name: str):
    """`open_spiel.python.observation` observer for a game, or None if this
    OpenSpiel build does not provide one. Per thread, because the observer
    owns a single reusable buffer (the backend serves Arena requests from
    a thread pool)."""
    cache = getattr(_OBSERVERS, "cache", None)
    if cache is None:
        cache = _OBSERVERS.cache = {}
    if name not in cache:
        try:
            from open_spiel.python.observation import make_observation

            cache[name] = make_observation(_pyspiel_game(name))
        except Exception:  # noqa: BLE001
            cache[name] = None
    return cache[name]


def _fen_to_board(fen: str) -> np.ndarray:
    placement = fen.split()[0]
    rows: list[list[int]] = []
    for rank in placement.split("/"):
        row: list[int] = []
        for ch in rank:
            if ch.isdigit():
                row.extend([0] * int(ch))
            else:
                row.append(_FEN_PIECE[ch])
        rows.append(row)
    return np.asarray(rows, dtype=np.int8)
class OpenSpielBoardGame(BoardGame):
    """Shared wrapper. Subclasses set `id`, `name`, `rows`, `cols`,
    `openspiel_name`."""

    openspiel_name: str
    win_length = 0

    def __init__(self) -> None:
        self._game = _pyspiel_game(self.openspiel_name)
        self._action_n = int(self._game.num_distinct_actions())
        self._obs_shape = tuple(int(x) for x in self._game.observation_tensor_shape())
        self._first_os_player = 0
        self._state = None
        self._board_cache = None
        self.reset()

    # Parsing FEN / Go text into `board` costs more than the engine step
    # itself, and search never looks at it — so it is computed on demand.
    @property
    def board(self) -> np.ndarray:
        if self._board_cache is None:
            self._board_cache = self._compute_board()
        return self._board_cache

    @board.setter
    def board(self, value: np.ndarray) -> None:
        self._board_cache = value

    @property
    def action_size(self) -> int:
        return self._action_n

    @property
    def input_planes(self) -> int:
        return int(self._obs_shape[0])

    def reset(self) -> np.ndarray:
        self._state = self._game.new_initial_state()
        while self._state.is_chance_node():
            action, _prob = self._state.chance_outcomes()[0]
            self._state.apply_action(action)
        self._first_os_player = int(self._state.current_player())
        self.current_player = 1
        self.done = False
        self.winner = None
        self.last_move = None
        self._sync_board()
        return self.encode()

    def legal_actions(self) -> list[int]:
        if self.done:
            return []
        return list(self._state.legal_actions())

    def encode(self) -> np.ndarray:
        # Terminal OpenSpiel states have current_player == −4, and
        # observation_tensor() then throws. MCTS/self-play never read the
        # encoding of a finished position (they already have the winner).
        if self._state.is_terminal():
            return np.zeros(self._obs_shape, dtype=np.float32)
        observer = _observer(self.openspiel_name)
        if observer is None:
            flat = np.asarray(self._state.observation_tensor(), dtype=np.float32)
            return flat.reshape(self._obs_shape)
        # ~50× faster than converting the Python list from
        # observation_tensor(): the C++ side writes straight into a numpy
        # buffer, which is reused — hence the copy.
        observer.set_from(self._state, self._state.current_player())
        return observer.tensor.reshape(self._obs_shape).copy()

    def step(self, action: int) -> tuple[np.ndarray, bool, int | None]:
        if self.done:
            raise ValueError("Game already finished")
        action = int(action)
        if action not in self._state.legal_actions():
            raise ValueError(f"Illegal OpenSpiel action: {action}")
        self.step_trusted(action)
        return self.encode(), self.done, self.winner

    def step_trusted(self, action: int) -> tuple[np.ndarray | None, bool, int | None]:
        """`step()` for an action already taken from `legal_actions()`:
        skips the legality re-check and does not build the encoding (the
        first return value is None) — the hot path of batched search."""
        self._state.apply_action(int(action))
        if self._state.is_terminal():
            self.done = True
            returns = self._state.returns()
            first_ret = float(returns[self._first_os_player])
            if first_ret > 0:
                self.winner = 1
            elif first_ret < 0:
                self.winner = -1
            else:
                self.winner = 0
            # Same "flip even on the terminal ply" convention as BoardGame
            # (see mcts.py) — not OpenSpiel's terminal player id (−4).
            self.current_player = -self.current_player
        else:
            os_p = int(self._state.current_player())
            self.current_player = 1 if os_p == self._first_os_player else -1
        self._sync_board()
        return None, self.done, self.winner

    def clone(self) -> BoardGame:
        cloned = self.__class__.__new__(self.__class__)
        cloned._game = self._game
        cloned._action_n = self._action_n
        cloned._obs_shape = self._obs_shape
        cloned._first_os_player = self._first_os_player
        cloned._state = self._state.clone()
        cloned._board_cache = None if self._board_cache is None else self._board_cache.copy()
        cloned.current_player = self.current_player
        cloned.done = self.done
        cloned.winner = self.winner
        cloned.last_move = self.last_move
        return cloned

    def _sync_board(self) -> None:
        self._board_cache = None

    def _compute_board(self) -> np.ndarray:
        return np.zeros((self.rows, self.cols), dtype=np.int8)

    def render_ascii(self, symbols: tuple[str, str, str] = (".", "X", "O")) -> str:
        fn = getattr(self._state, "debug_string", None) or getattr(self._state, "to_string", None)
        if fn is not None:
            return str(fn())
        return super().render_ascii(symbols)


class Chess(OpenSpielBoardGame):
    id = "chess"
    name = "Chess"
    rows = 8
    cols = 8
    openspiel_name = "chess"

    def _compute_board(self) -> np.ndarray:
        return _fen_to_board(self._state.board().to_fen())

def _encode_hwc(enc: np.ndarray, rows: int, cols: int) -> np.ndarray:
    """`BoardGame.encode()` is channel-first `(C, H, W)`; Gym image nets
    here expect channel-last `(H, W, C)` uint8 (see `SmallCNN`)."""
    arr = np.asarray(enc, dtype=np.float32)
    if arr.ndim != 3:
        raise ValueError(f"board encode must be 3D, got {arr.shape}")
    if arr.shape[0] == rows and arr.shape[1] == cols:
        hwc = arr
    elif arr.shape[1] == rows and arr.shape[2] == cols:
        hwc = np.transpose(arr, (1, 2, 0))
    else:
        hwc = np.transpose(arr, (1, 2, 0))
    return np.clip(np.round(hwc * 255.0), 0, 255).astype(np.uint8)


_CHESS_LETTERS = {1: "P", 2: "N", 3: "B", 4: "R", 5: "Q", 6: "K"}
_LIGHT_SQ = (232, 213, 181)
_DARK_SQ = (181, 136, 99)
_WHITE_TOKEN = (244, 241, 234)
_BLACK_TOKEN = (32, 30, 28)
_WHITE_INK = (36, 32, 28)
_BLACK_INK = (244, 241, 234)
_CHESS_MATERIAL_VALUES = np.asarray([0, 1, 3, 3, 5, 9, 0], dtype=np.float32)
_CHESS_TOTAL_MATERIAL = 39.0


def _chess_material_for(board: np.ndarray, player: int) -> float:
    pieces = np.asarray(board, dtype=np.int32)
    values = _CHESS_MATERIAL_VALUES[np.clip(np.abs(pieces), 0, 6)]
    signs = np.sign(pieces) * int(player)
    return float(np.sum(values * signs))


def _piece_font(size: int):
    from PIL import ImageFont

    candidates = (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
    )
    for path in candidates:
        try:
            return ImageFont.truetype(path, size=size)
        except OSError:
            continue
    try:
        return ImageFont.load_default(size=size)
    except TypeError:
        return ImageFont.load_default()


def render_board_rgb(board: np.ndarray, game_id: str = "") -> np.ndarray:
    """Human-facing RGB frame for Training Monitor GIFs.

    Occupied cells used to be solid white/black rectangles — fine for
    Tic-Tac-Toe, meaningless for chess (every piece looked like a square).
    Chess draws lettered tokens (P/N/B/R/Q/K); the ±1 games draw stones.
    """
    from PIL import Image, ImageDraw

    grid = np.asarray(board, dtype=np.int32)
    if grid.ndim != 2:
        raise ValueError(f"board must be 2D, got {grid.shape}")
    rows, cols = int(grid.shape[0]), int(grid.shape[1])
    chess = game_id == "chess" or (rows == 8 and cols == 8 and int(np.max(np.abs(grid))) > 1)
    cell = 40 if rows >= 8 else 52
    img = Image.new("RGB", (cols * cell, rows * cell), (21, 21, 32))
    draw = ImageDraw.Draw(img)

    if chess or game_id in ("gomoku", "go_9x9"):
        light, dark = _LIGHT_SQ, _DARK_SQ
    elif game_id == "connect_four":
        light = dark = (30, 90, 170)
    else:
        light, dark = (52, 52, 68), (36, 36, 48)

    for row in range(rows):
        for col in range(cols):
            x0, y0 = col * cell, row * cell
            fill = light if (row + col) % 2 == 0 else dark
            draw.rectangle((x0, y0, x0 + cell - 1, y0 + cell - 1), fill=fill)

    font = _piece_font(max(14, cell * 11 // 20)) if chess else None
    for row in range(rows):
        for col in range(cols):
            value = int(grid[row, col])
            if value == 0:
                continue
            cx = col * cell + cell // 2
            cy = row * cell + cell // 2
            white = value > 0
            if chess:
                radius = cell // 2 - 4
                token = _WHITE_TOKEN if white else _BLACK_TOKEN
                ink = _WHITE_INK if white else _BLACK_INK
                draw.ellipse(
                    (cx - radius, cy - radius, cx + radius, cy + radius),
                    fill=token,
                    outline=(70, 58, 42),
                )
                glyph = _CHESS_LETTERS.get(min(abs(value), 6), "?")
                bbox = draw.textbbox((0, 0), glyph, font=font)
                tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
                draw.text(
                    (cx - tw / 2 - bbox[0], cy - th / 2 - bbox[1] - 1),
                    glyph,
                    font=font,
                    fill=ink,
                )
            else:
                radius = cell // 2 - 5
                fill = _WHITE_TOKEN if white else _BLACK_TOKEN
                outline = (40, 40, 48) if white else (210, 210, 216)
                draw.ellipse(
                    (cx - radius, cy - radius, cx + radius, cy + radius),
                    fill=fill,
                    outline=outline,
                    width=2,
                )
    return np.asarray(img, dtype=np.uint8)


class ChessSelfPlayEnv(gym.Env):
    """Один эпизод Gym = одна партия, одна и та же политика ходит за обе стороны."""

    metadata = {"render_modes": ["rgb_array"], "render_fps": 4, "two_player": True}

    def __init__(self, render_mode: str | None = None) -> None:
        super().__init__()
        self.game_id = "chess"
        self.game = Chess()
        self.two_player = True
        self.render_mode = render_mode
        self._steps = 0
        planes = int(self.game.input_planes)
        h, w = int(self.game.rows), int(self.game.cols)
        self.observation_space = spaces.Box(0, 255, (h, w, planes), dtype=np.uint8)
        self.action_space = spaces.Discrete(int(self.game.action_size))

    def _obs(self) -> np.ndarray:
        return _encode_hwc(self.game.encode(), self.game.rows, self.game.cols)

    def action_masks(self) -> np.ndarray:
        mask = np.zeros(int(self.action_space.n), dtype=np.int8)
        if not self.game.done:
            mask[np.asarray(self.game.legal_actions(), dtype=np.int64)] = 1
        return mask

    def _info(self) -> dict:
        return {
            "action_mask": self.action_masks(),
            "two_player": True,
            "board": self.game.board_list(),
            "current_player": int(self.game.current_player),
            "winner": (
                None if self.game.winner is None else int(self.game.winner)
            ),
            "game_done": bool(self.game.done),
        }

    def reset(self, *, seed: int | None = None, options: dict | None = None) -> tuple[np.ndarray, dict]:
        super().reset(seed=seed)
        self.game.reset()
        self._steps = 0
        return self._obs(), self._info()

    def step(self, action: int) -> tuple[np.ndarray, float, bool, bool, dict]:
        action = int(action)
        if self.game.done:
            return self._obs(), 0.0, True, False, self._info()

        legal = self.game.legal_actions()
        if action not in legal:
            # Search/warmup should never do this when the mask is honoured.
            # Treat as an immediate loss for the mover so a VectorEnv worker
            # doesn't die on `BoardGame.step`'s ValueError.
            mover = int(self.game.current_player)
            self.game.done = True
            self.game.winner = -mover
            self.game.current_player = -mover
            info = self._info()
            info["illegal_action"] = True
            return self._obs(), -1.0, True, False, info

        mover = int(self.game.current_player)
        material_before = _chess_material_for(self.game.board, mover) if self.game_id == "chess" else 0.0
        _enc, done, winner = self.game.step(action)
        self._steps += 1
        # `step` flips `current_player` even on the terminal ply, so the
        # mover is the opposite of whoever is to-move now.
        if self.game_id == "chess" and not done:
            # Pure terminal chess reward gives a randomly initialized
            # model virtually no signal: most self-play games terminate
            # by repetition/ply limit as draws.  A normalized material
            # delta is a zero-sum, mover-relative transition reward; with
            # the Zero algorithms' -gamma backup, our captures are
            # positive and the opponent's captures are negative.
            material_after = _chess_material_for(self.game.board, mover)
            reward = (material_after - material_before) / _CHESS_TOTAL_MATERIAL
        elif not done:
            reward = 0.0
        elif winner in (0, None):
            reward = 0.0
        else:
            reward = 1.0 if int(winner) == mover else -1.0
        return self._obs(), float(reward), bool(done), False, self._info()

    def render(self) -> np.ndarray | None:
        if self.render_mode != "rgb_array":
            return None
        return render_board_rgb(self.game.board_list(), self.game_id)


MAX_EPISODE_STEPS = 512


def make_chess_env(render_mode: str | None = None) -> gym.Env:
    return gym.wrappers.TimeLimit(
        ChessSelfPlayEnv(render_mode=render_mode),
        max_episode_steps=MAX_EPISODE_STEPS,
    )


_probe = make_chess_env()
_obs, _info = _probe.reset(seed=SEED)
print(
    "chess obs", tuple(_obs.shape), _obs.dtype,
    "actions", int(_probe.action_space.n),
    "legal", int(_info["action_mask"].sum()),
)
_probe.close()


In [ ]:
"""Кодирование наблюдений, свёрточная сеть и сбор параллельных сред.

`SmallCNN` рассчитан на маленькие поля вроде 8×8. Векторные хелперы ниже
сохраняют контракт SAME_STEP: после конца партии политика видит уже новую
позицию, а настоящий терминальный кадр лежит в `info["final_obs"]`.
"""
def is_image_space(space: gym.Space) -> bool:
    """(H, W, C) or (C, H, W) uint8-ish Box — treat as a CNN input.

    The `min(shape) <= 4` branch is the usual Atari/MiniGrid case (a
    channel axis of 1–4). Board-game tensors such as chess `(8, 8, 20)`
    have more planes than that, so a small spatial H×W with a moderate
    channel count is also accepted as an image (channel-last).
    """
    if not isinstance(space, gym.spaces.Box) or len(space.shape) != 3:
        return False
    if min(space.shape) <= 4:
        return True
    height, width, channels = (int(dim) for dim in space.shape)
    return 3 <= height <= 32 and 3 <= width <= 32 and 1 <= channels <= 128


def obs_flat_dim(space: gym.Space) -> int:
    """Flat feature count fed to an MLP (ignored for image spaces)."""
    if isinstance(space, gym.spaces.Discrete):
        return int(space.n)
    return int(np.prod(space.shape))


def obs_to_array(obs, space: gym.Space) -> np.ndarray:
    """Single (non-batched) observation -> numpy array ready for stacking.

    - `Discrete` -> one-hot float32 vector (SB3 does the same by default).
    - Image `Box` -> kept as (H, W, C) uint8/float32, normalized in the CNN.
    - Everything else (vector `Box`) -> flat float32 vector.
    """
    if isinstance(space, gym.spaces.Discrete):
        vec = np.zeros(int(space.n), dtype=np.float32)
        vec[int(obs)] = 1.0
        return vec
    arr = np.asarray(obs)
    if is_image_space(space):
        return arr.astype(np.float32)
    return arr.reshape(-1).astype(np.float32)


def obs_batch_to_array(obs_list: list, space: gym.Space) -> np.ndarray:
    """`obs_list`: a length-N list of raw (non-batched) per-lane
    observations, e.g. from `vec_reset` / `vec_step` — stacks `obs_to_array(...)` over each into one `(N, ...)`
    float32 array. `N=1` (the `num_envs=1` default) degenerates to a
    length-1 batch of exactly what `obs_to_array` alone would return."""
    return np.stack([obs_to_array(o, space) for o in obs_list])


def action_to_env(action: np.ndarray, action_space: gym.Space):
    """Native nets always produce a small numpy array per action; env.step
    wants a Python int for `Discrete`, a properly-shaped/clamped array for
    `Box`."""
    if isinstance(action_space, gym.spaces.Discrete):
        return int(np.asarray(action).reshape(-1)[0])
    low, high = action_space.low, action_space.high
    return np.clip(np.asarray(action, dtype=np.float32).reshape(action_space.shape), low, high)

class MLPExtractor(nn.Module):
    def __init__(self, in_dim: int, hidden: tuple[int, ...] = (64, 64)) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        last = in_dim
        for h in hidden:
            layers += [nn.Linear(last, h), nn.Tanh()]
            last = h
        self.net = nn.Sequential(*layers)
        self.out_dim = last

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class NatureCNN(nn.Module):
    """Mnih et al. (2015) DQN conv stack. Input is (B, H, W, C) uint8/float
    (channel-last, matching Gym's raw frames) — normalized and permuted to
    (B, C, H, W) internally."""

    def __init__(self, obs_shape: tuple[int, int, int]) -> None:
        super().__init__()
        h, w, c = obs_shape
        self.conv = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            n_flatten = self.conv(torch.zeros(1, c, h, w)).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten, 512), nn.ReLU())
        self.out_dim = 512

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 3, 1, 2).float() / 255.0
        return self.linear(self.conv(x))


class SmallCNN(nn.Module):
    """Lighter conv stack for small (roughly <50px) image observations —
    e.g. the egocentric crop in `VisualMemoryMazeEnv`. `NatureCNN`'s
    aggressive stride-4/stride-2/stride-1 stack was tuned for 84x84 Atari
    frames; on anything much smaller than that it either errors out
    (stride-3 kernel bigger than what's left of the feature map) or
    collapses to a 1x1 spatial map before the final conv layer, throwing
    away exactly the spatial structure a small egocentric view is mostly
    made of. This stack uses `padding` and gentler strides so the spatial
    size only roughly halves once, keeping meaningfully more of it."""

    def __init__(self, obs_shape: tuple[int, int, int]) -> None:
        super().__init__()
        h, w, c = obs_shape
        self.conv = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            n_flatten = self.conv(torch.zeros(1, c, h, w)).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten, 256), nn.ReLU())
        self.out_dim = 256

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 3, 1, 2).float() / 255.0
        return self.linear(self.conv(x))


# Smallest side length `NatureCNN`'s stride-4/stride-2/stride-1 stack can
# accept without erroring (empirically ~38px) — anything under this
# comfortably-larger threshold routes to `SmallCNN` instead. Atari/CarRacing
# frames (84px+) are always well above it either way.
_NATURE_CNN_MIN_SIDE = 50


def build_feature_extractor(observation_space: gym.Space) -> nn.Module:
    if is_image_space(observation_space):
        shape = tuple(observation_space.shape)
        if min(shape[0], shape[1]) < _NATURE_CNN_MIN_SIDE:
            return SmallCNN(shape)
        return NatureCNN(shape)
    return MLPExtractor(obs_flat_dim(observation_space))

class ObsEncoder(nn.Module):
    """Thin wrapper around `build_feature_extractor` that also remembers
    what it's encoding — every world model needs both the feature
    extractor itself and (for the decoder / reconstruction-loss target on
    the other side) whether the observation is an image and what its exact
    shape is."""

    def __init__(self, observation_space: gym.Space) -> None:
        super().__init__()
        self.observation_space = observation_space
        self.is_image = is_image_space(observation_space)
        self.net = build_feature_extractor(observation_space)
        self.out_dim = self.net.out_dim
        self.obs_shape = tuple(int(d) for d in observation_space.shape) if self.is_image else (obs_flat_dim(observation_space),)

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.net(obs)

def is_vector_env(env: Any) -> bool:
    return isinstance(env, VectorEnv)


def num_envs_of(env: Any) -> int:
    return int(env.num_envs) if is_vector_env(env) else 1


def obs_space(env: Any) -> gym.Space:
    """The *single* (unbatched) observation space regardless of whether
    `env` is a plain env or a `VectorEnv` — this is what network input
    dims/`obs_to_array` must always be built from, never the vector env's
    own (batched) `observation_space`."""
    return env.single_observation_space if is_vector_env(env) else env.observation_space


def action_space(env: Any) -> gym.Space:
    """See `obs_space` — the single (unbatched) action space."""
    return env.single_action_space if is_vector_env(env) else env.action_space


def _split_vector_infos(infos: dict[str, Any], num_envs: int) -> list[dict[str, Any]]:
    """`VectorEnv.step`'s `infos` is a dict-of-arrays (one array per key
    that *any* lane's info dict had), not a list of per-lane dicts — and
    any key whose per-lane value is itself a dict (e.g. `Monitor`'s
    `"episode"` key, present only on the step a lane's episode ends) gets
    aggregated the same way, recursively, rather than turned into a plain
    array. Each key also gets a same-shape `f"_{key}"` boolean mask
    alongside it marking which lanes actually had that key this step (the
    rest hold an arbitrary filler value that must not leak into a lane
    that never reported it). Recurses through that nested structure to
    rebuild the length-`num_envs` list of plain per-lane dicts every
    caller in this app actually wants."""
    result: list[dict[str, Any]] = [{} for _ in range(num_envs)]
    for key, value in infos.items():
        if key.startswith("_"):
            continue
        mask = infos.get(f"_{key}")
        if isinstance(value, dict):
            per_lane = _split_vector_infos(value, num_envs)
        else:
            per_lane = list(value)
        for i in range(num_envs):
            if mask is None or mask[i]:
                result[i][key] = per_lane[i]
    return result


def vec_reset(
    env: Any, seed: int | None = None, *, return_info: bool = False,
) -> list[Any] | tuple[list[Any], list[dict]]:
    """Returns a length-`num_envs_of(env)` list of raw (non-batched, one
    per lane) observations — same shape of thing whether `env` is a plain
    env (`[obs]`) or a `VectorEnv` (its batched array/tuple, unpacked along
    axis 0). Every lane gets a distinct seed (`seed + i`) so `num_envs>1`
    doesn't just replay `num_envs` identical trajectories.

    `return_info=True` also returns the per-lane info dicts (needed for
    board-game `action_mask` on the first search of an episode).
    """
    if is_vector_env(env):
        seeds = [seed + i for i in range(env.num_envs)] if seed is not None else None
        obs, info = env.reset(seed=seeds)
        obs_list = list(obs)
        infos = _split_vector_infos(info, env.num_envs)
    else:
        obs, info = env.reset(seed=seed)
        obs_list, infos = [obs], [info]
    if return_info:
        return obs_list, infos
    return obs_list


def vec_step(
    env: Any, actions: Sequence[Any],
) -> tuple[list[Any], np.ndarray, np.ndarray, np.ndarray, list[dict]]:
    """`actions`: length-`num_envs_of(env)` sequence of *unbatched* per-lane
    actions (whatever `action_to_env(...)` produces for the single action
    space — a plain `int` for `Discrete`, a shaped `np.ndarray` for `Box`).
    Returns `(obs_list, rewards, terminated, truncated, infos)`, every one
    of them length-N — `obs_list` a plain list (see `vec_reset`), the rest
    numpy arrays, `infos` a length-N list of per-lane dicts (not
    `VectorEnv`'s own dict-of-arrays shape, so callers never need to
    branch on vector-vs-not to read a lane's info).

    Vector envs are constructed with `SAME_STEP` autoreset: a terminal
    call returns the freshly reset observation for the next policy call,
    while the real terminal observation/info are preserved as
    `info["final_obs"]`/`info["final_info"]`. This avoids NEXT_STEP's
    otherwise unavoidable dummy iteration where the supplied action is
    ignored and callers accidentally record `terminal -> reset` as a
    zero-reward transition. Plain envs are normalized to the same
    contract below.
    """
    if is_vector_env(env):
        obs, rewards, terminated, truncated, infos = env.step(np.asarray(actions))
        num_envs = env.num_envs
        per_lane_infos = _split_vector_infos(infos, num_envs)
        # SAME_STEP nests the terminal env's info under `final_info` and
        # uses the top level for reset info. Preserve the explicit nested
        # field, but also expose terminal keys at the top level to retain
        # vec_step's historical per-lane info contract (notably Monitor's
        # `episode` entry).
        for lane_info in per_lane_infos:
            final_info = lane_info.get("final_info")
            if isinstance(final_info, dict):
                for key, value in final_info.items():
                    lane_info.setdefault(key, value)
        return (
            list(obs),
            np.asarray(rewards, dtype=np.float32),
            np.asarray(terminated, dtype=bool),
            np.asarray(truncated, dtype=bool),
            per_lane_infos,
        )
    obs, reward, terminated, truncated, info = env.step(actions[0])
    if terminated or truncated:
        final_obs, final_info = obs, info
        obs, reset_info = env.reset()
        # Reset keys win for anything both dicts have — except we still
        # expose the terminal payload under `final_*`. `action_mask` on a
        # board-game env is the *next* position's legal set (the one the
        # following search must use); taking it from `final_info` would
        # leave an empty terminal mask on the freshly reset observation.
        info = {**reset_info, **final_info, "final_obs": final_obs, "final_info": final_info}
        if "action_mask" in reset_info:
            info["action_mask"] = reset_info["action_mask"]
    return (
        [obs],
        np.array([reward], dtype=np.float32),
        np.array([terminated], dtype=bool),
        np.array([truncated], dtype=bool),
        [info],
    )

MAX_INTERIOR_ACTIONS = 32
LARGE_ACTION_SPACE = 64

ZERO_SELFPLAY_ALGORITHMS = (
    "efficientzero",
    "unizero",
    "researchimzero",
    "latentimzero",
)


def is_two_player_env(env: Any) -> bool:
    """True when the (possibly wrapped / vectorized) env is alternating
    two-player self-play — see `BoardGameSelfPlayEnv.two_player`."""
    candidates: list[Any] = [env]
    if is_vector_env(env):
        inner = getattr(env, "unwrapped", None)
        if inner is not None:
            candidates.append(inner)
    seen: set[int] = set()
    for start in candidates:
        cur = start
        while cur is not None and id(cur) not in seen:
            seen.add(id(cur))
            if bool(getattr(cur, "two_player", False)):
                return True
            meta = getattr(cur, "metadata", None) or {}
            if meta.get("two_player"):
                return True
            nxt = getattr(cur, "env", None)
            if nxt is None or nxt is cur:
                unwrapped = getattr(cur, "unwrapped", None)
                cur = unwrapped if unwrapped is not cur else None
            else:
                cur = nxt
    return False


def attach_two_player(algo: Any, env: Any) -> None:
    """Set `two_player` / `search_discount` on a Zero algorithm. Call after
    `algo.gamma` is assigned."""
    algo.two_player = is_two_player_env(env)
    algo.search_discount = -float(algo.gamma) if algo.two_player else float(algo.gamma)


def maybe_enable_two_player_from_infos(algo: Any, infos: list[dict[str, Any]]) -> None:
    """VectorEnv workers don't always expose `two_player` on the parent;
    the Gym wrapper also puts it in `info` every step."""
    if getattr(algo, "two_player", False):
        return
    if any(bool(info.get("two_player")) for info in infos):
        algo.two_player = True
        algo.search_discount = -float(algo.gamma)


def terminal_self_play_outcome(
    info: dict[str, Any],
    terminated: bool,
    truncated: bool,
) -> int | None:
    """Return the board-game winner for a completed real game.

    ``None`` deliberately distinguishes a time-limit/unfinished trajectory
    from a genuine draw (``0``).  Vector envs put terminal data under
    ``final_info`` after SAME_STEP autoreset.
    """
    if not terminated or truncated:
        return None
    terminal_info = info.get("final_info")
    if not isinstance(terminal_info, dict):
        terminal_info = info
    winner = terminal_info.get("winner")
    return int(winner) if winner in (-1, 0, 1) else None


def self_play_replay_capacity(requested: int, num_lanes: int, two_player: bool) -> int:
    """Keep enough completed games for parallel self-play diversity.

    A capacity smaller than the vector width churns almost the entire
    replay on every wave of completed games. Four games per lane is a
    conservative floor; compact float16 storage remains responsible for
    keeping large chess policy targets affordable.
    """
    requested = max(1, int(requested))
    return max(requested, 4 * max(1, int(num_lanes))) if two_player else requested


def self_play_eviction_index(
    outcomes: list[int | None],
    capacity: int,
    reserve_fraction: float = 0.25,
) -> int:
    """Choose an overflow eviction while retaining a small decisive archive.

    FIFO remains the default. For outcome-labelled self-play, up to a
    quarter of replay is protected for rare decisive games; stale decisive
    data can still age out once that reserve is full.
    """
    if len(outcomes) <= capacity or not any(outcome is not None for outcome in outcomes):
        return 0
    decisive = sum(outcome in (-1, 1) for outcome in outcomes)
    reserve = max(1, int(round(max(1, capacity) * reserve_fraction)))
    if decisive <= reserve:
        for index, outcome in enumerate(outcomes):
            if outcome == 0:
                return index
    return 0


def record_self_play_outcomes(
    algo: Any,
    infos: list[dict[str, Any]],
    terminated: np.ndarray,
    truncated: np.ndarray,
) -> None:
    """Accumulate board-game outcomes independently from shaped rewards."""
    if not getattr(algo, "two_player", False):
        return
    if not hasattr(algo, "_self_play_outcomes"):
        algo._self_play_outcomes = {
            "games": 0,
            "decisive": 0,
            "draws": 0,
            "truncations": 0,
            "first_player_wins": 0,
            "second_player_wins": 0,
            "illegal_actions": 0,
        }
    stats = algo._self_play_outcomes
    for lane in np.flatnonzero(np.asarray(terminated) | np.asarray(truncated)):
        stats["games"] += 1
        lane_info = infos[int(lane)]
        terminal_info = lane_info.get("final_info")
        if not isinstance(terminal_info, dict):
            terminal_info = lane_info
        if bool(terminal_info.get("illegal_action")):
            stats["illegal_actions"] += 1
        if bool(truncated[int(lane)]):
            stats["truncations"] += 1
            continue
        winner = terminal_self_play_outcome(
            lane_info,
            bool(terminated[int(lane)]),
            bool(truncated[int(lane)]),
        )
        if winner in (1, -1):
            stats["decisive"] += 1
            key = "first_player_wins" if int(winner) == 1 else "second_player_wins"
            stats[key] += 1
        else:
            stats["draws"] += 1


def self_play_outcome_metrics(algo: Any) -> dict[str, float]:
    stats = getattr(algo, "_self_play_outcomes", None)
    if not stats:
        return {}
    games = max(1, int(stats["games"]))
    decisive = max(1, int(stats["decisive"]))
    return {
        "self_play_games": float(stats["games"]),
        "self_play_decisive_games": float(stats["decisive"]),
        "self_play_draws": float(stats["draws"]),
        "self_play_truncations": float(stats["truncations"]),
        "self_play_first_player_wins": float(stats["first_player_wins"]),
        "self_play_second_player_wins": float(stats["second_player_wins"]),
        "self_play_illegal_actions": float(stats["illegal_actions"]),
        "self_play_decisive_rate": float(stats["decisive"]) / games,
        "self_play_draw_rate": float(stats["draws"]) / games,
        "self_play_truncation_rate": float(stats["truncations"]) / games,
        "self_play_first_player_win_rate_decisive": (
            float(stats["first_player_wins"]) / decisive
            if stats["decisive"]
            else 0.0
        ),
    }


def action_mask_from_info(info: dict[str, Any], n_actions: int) -> np.ndarray | None:
    mask = info.get("action_mask")
    if mask is None:
        return None
    arr = np.asarray(mask).reshape(-1)
    if arr.size != n_actions:
        return None
    return (arr > 0).astype(np.int8)


def masks_from_infos(infos: list[dict[str, Any]], n_actions: int) -> np.ndarray | None:
    masks = [action_mask_from_info(info, n_actions) for info in infos]
    if not masks or any(m is None for m in masks):
        return None
    return np.stack(masks)


def mask_from_unwrapped_env(env: Any, n_actions: int) -> np.ndarray | None:
    """Best-effort legal mask for `predict()`, which has no `info` dict."""
    cur = env
    seen: set[int] = set()
    while cur is not None and id(cur) not in seen:
        seen.add(id(cur))
        fn = getattr(cur, "action_masks", None)
        if callable(fn):
            mask = np.asarray(fn()).reshape(-1)
            if mask.size == n_actions:
                return (mask > 0).astype(np.int8)
        nxt = getattr(cur, "env", None)
        cur = nxt if nxt is not cur else getattr(cur, "unwrapped", None)
    return None


def sample_masked_action(action_space: Any, mask: np.ndarray | None) -> int:
    if mask is None:
        return int(action_space.sample())
    legal = np.flatnonzero(mask)
    if legal.size == 0:
        return int(action_space.sample())
    return int(np.random.choice(legal))


def uniform_legal_policy(n_actions: int, mask: np.ndarray | None) -> np.ndarray:
    policy = np.zeros(n_actions, dtype=np.float32)
    if mask is None:
        policy[:] = 1.0 / max(n_actions, 1)
        return policy
    legal = np.flatnonzero(mask)
    if legal.size == 0:
        policy[:] = 1.0 / max(n_actions, 1)
        return policy
    policy[legal] = 1.0 / legal.size
    return policy


def discrete_expand_slots(
    logits: np.ndarray,
    *,
    action_mask: np.ndarray | None = None,
    max_children: int | None = None,
    as_logits: bool = True,
) -> tuple[np.ndarray, list[int]]:
    """Pick the children a discrete search node should expand.

    `logits` is the raw policy head `(n_actions,)`. When `as_logits` is
    True the returned priors stay in log-space (Gumbel-Top-k); otherwise
    they are a softmax over the selected actions (PUCT).
    """
    logits = np.asarray(logits, dtype=np.float64).reshape(-1)
    n = int(logits.shape[0])
    if action_mask is not None:
        legal = np.flatnonzero(np.asarray(action_mask).reshape(-1) > 0)
        if legal.size == 0:
            legal = np.arange(n)
    else:
        legal = np.arange(n)
    if max_children is not None and legal.size > max_children:
        scores = logits[legal]
        keep = np.argpartition(-scores, max_children - 1)[:max_children]
        legal = legal[keep]
    child_logits = logits[legal]
    action_ids = [int(a) for a in legal.tolist()]
    if as_logits:
        return child_logits, action_ids
    shifted = child_logits - child_logits.max()
    exp = np.exp(shifted)
    return exp / max(float(exp.sum()), 1e-12), action_ids


def interior_max_children(n_actions: int) -> int | None:
    if n_actions > LARGE_ACTION_SPACE:
        return MAX_INTERIOR_ACTIONS
    return None


def scatter_policy(child_policy: np.ndarray, action_ids: list[int], n_actions: int) -> np.ndarray:
    full = np.zeros(n_actions, dtype=np.float32)
    for prob, action_id in zip(child_policy, action_ids):
        full[int(action_id)] += float(prob)
    total = float(full.sum())
    if total > 0:
        full /= total
    return full


def child_action_ids(node: Any) -> list[int]:
    ids: list[int] = []
    for i, child in enumerate(node.children):
        action = getattr(child, "candidate_action", None)
        ids.append(int(action) if action is not None else i)
    return ids


def discrete_policy_target(node: Any, child_policy: np.ndarray, n_actions: int) -> np.ndarray:
    ids = child_action_ids(node)
    if ids == list(range(len(node.children))) and len(node.children) == n_actions:
        return np.asarray(child_policy, dtype=np.float32)
    return scatter_policy(child_policy, ids, n_actions)


def discrete_env_action(node: Any, child_index: int) -> int:
    child = node.children[child_index]
    action = getattr(child, "candidate_action", None)
    return int(action) if action is not None else int(child_index)


def chosen_discrete_action(leaf: Any, parent: Any) -> int:
    action = getattr(leaf, "candidate_action", None)
    if action is not None:
        return int(action)
    return int(parent.children.index(leaf))

venv_obs_space = obs_space
venv_action_space = action_space


In [ ]:
"""EfficientZero V2 (Wang, Liu, Ye, You & Gao, ICML 2024).

EfficientZero V2 (Wang, Liu, Ye, You & Gao, ICML 2024 Spotlight -
https://arxiv.org/abs/2403.00564), built on EfficientZero (Ye et al.,
NeurIPS 2021 - https://arxiv.org/abs/2111.00210) and MuZero (Schrittwieser
et al., 2020 - https://arxiv.org/abs/1911.08265). Like every other
model-based algorithm in this app it learns a world model from real
experience - but where Dreamer imagines whole rollouts for an actor-critic
and PETS/MBPO plan with a CEM search over an ensemble, EfficientZero
learns a *compact* representation/dynamics/prediction triple (a la MuZero)
and plans with a small, budgeted tree search at every real step, using
that search's own output both to pick the action and as the training
target for the policy/value heads.

Four MuZero-family ingredients carried over from EfficientZero (V1 and V2):
- Representation `h(obs) -> s`, dynamics `g(s, a) -> s', reward` and
  prediction `f(s) -> (policy, value)` networks - the same three-network
  split as MuZero itself, trained *only* through what the search touches
  (no pixel/vector reconstruction loss anywhere, unlike Dreamer/World
  Models' decoders) plus one extra self-supervised term below.
- Temporal consistency ("self-supervised consistency loss", Ye et al.
  2021, SimSiam-style): the dynamics net's predicted next latent is
  pushed towards the representation net's *real* encoding of the actual
  next observation, through an asymmetric projector/predictor pair with a
  stop-gradient on the real-encoding branch, *and* `BatchNorm1d` on both
  MLPs' hidden layer (`_Projector`/`_Predictor` - stop-gradient alone
  still collapses in practice per Chen & He, 2021's own ablation; BN is
  what actually keeps it from happening) - this is what keeps `s`
  predictive of the real observation instead of collapsing to whatever
  is easiest for the reward/value heads alone.
- Value prefix: the dynamics net's reward head is an LSTM carried across
  one training unroll, predicting the *cumulative* real reward since the
  unroll's start at each step rather than each step's reward in
  isolation - reduces compounding one-step reward-prediction error over
  a short unroll, exactly like the paper.
- Categorical value/reward ("scaling and squashing", MuZero Appendix F):
  both the value head and the value-prefix head predict a *distribution*
  over a fixed support (`2 * value_support_size + 1` bins) in a
  `signed_hyperbolic`-transformed scale, trained with cross-entropy
  against a two-hot target, instead of a single scalar regressed with
  MSE. This is not decorative - a plain scalar+MSE head's loss is
  quadratic in the *raw* reward/value scale, so any env with an
  occasional large or heavy-tailed reward (a boss kill, a big score jump,
  a rare bonus) makes that one minibatch's loss - and its gradient -
  explode relative to every ordinary step, exactly the "value_loss
  spikes" this app's own EfficientZero V2 runs show without it. The
  hyperbolic transform compresses that same raw range logarithmically
  *before* the fixed-width bins see it, so every bin (and therefore the
  cross-entropy loss/gradient) stays boundedly-scaled regardless of how
  large one particular target happens to be - the categorical head can
  represent a huge value accurately without blowing up training the way
  a scalar head chasing the same value with squared error would.
- Gumbel search (Danihelka et al., 2021 - the search EfficientZero V2
  itself reuses for its discrete case): sample a small set of candidate
  root actions via the Gumbel-Top-k trick, then narrow that set with
  Sequential Halving instead of exhaustively visiting a huge tree -
  cheap enough to run *every* real step, unlike classic PUCT-MCTS.

EfficientZero V2's own headline addition - supporting continuous action
spaces at all, which Gumbel search alone cannot (it needs a finite
candidate set) - is here too: root candidates are instead *sampled*, half
from the current Gaussian policy and half from a deliberately wider
version of it (the paper's `A_{S1}`/`A_{S2}` split), then the same
Sequential-Halving bandit picks a winner among them by simulated return.

The search itself (`_SearchNode`/`search()` below) is a *real* Gumbel
search tree - not a single-path lookahead - following "Policy improvement
by planning with Gumbel" (Danihelka, Guez, Schrittwieser & Silver, ICLR
2022, https://openreview.net/pdf?id=bERaNdoegnO), the search EfficientZero
V2 itself builds on: every one of `num_simulations` simulations descends
an actual tree by one edge per level (root: equal-visit round-robin over
the surviving top-`num_top_actions` candidates; every deeper node: the
paper's improved-policy-minus-visit-fraction rule, both using the same
`v_mix`/"completed Q" backup - Appendix D - so even *unvisited* children
get a sensible Q via their parent's value-mixed estimate instead of being
silently ignored), expands exactly one new leaf per lane via one batched
dynamics+prediction forward pass, and backs the resulting value up the
whole path with proper visit counts - genuinely the same mechanism the
paper (and EfficientZero V2's own `ez/mcts/py_mcts.py`) uses, just in pure
Python/PyTorch instead of their compiled `ctree_v2` (fine here: this app's
envs and `num_simulations` budgets are far smaller than Atari-with-a-
compiled-tree territory). Final action/policy target come from the same
formulas as the paper: Sequential Halving's surviving top action, and the
softmax of (root priors + transformed completed Qs) as the "improved
policy" distilled into the policy head at training time.

`training.num_envs > 1` collects real experience from that many parallel
env lanes (`AsyncVectorEnv`, one subprocess worker each), same as
Dreamer/World Models (Ha) - and here it's not just "run N independent
searches", it's a genuinely *batched* search: `search()` takes the whole
`(N, *obs_shape)` batch of lanes at once and runs all N trees' simulations
in lockstep, so every simulation's leaf-expansion dynamics/prediction call
is one batched `(N, ...)` forward pass, not N separate ones. Each lane
still gets its own tree/root/`_MinMaxStats` and its own slot in
`_EfficientZeroBuffer` (`add(..., lane=i)`) so lanes' episodes never splice
into one another. This is exactly Dreamer's own "single biggest
real-world speedup lever" story: the *search* is roughly the same total
compute either way (same handful of tiny batched forward passes, batch
dimension N instead of 1), but real-experience *collection* - the part
every model-based algorithm here is fundamentally bottlenecked on until
the model is decent - now gets `num_envs` real transitions per wall-clock
env.step() instead of one.

Two more pieces ported from the official repo's own training recipe
(`ez/agents/base.py`/`ez/config/exp/atari.yaml`) rather than staying with
a "single fixed target, uniform replay" simplification:
- **Prioritized replay** (Schaul et al., 2016, `priority.use_priority:
  True` in every one of the reference's own configs): `_EfficientZeroBuffer`
  samples transitions proportional to `|value_prediction - value_target|
  ** priority_alpha` from the last time each was trained on (the
  reference's own `fresh_priority`) instead of uniformly, with the
  standard importance-sampling correction (`priority_beta`) applied to
  the loss so this doesn't bias the gradient - just refocuses *when* each
  transition's gradient contribution lands.
- **Reanalyze + `value_target: 'max'`**: the reference's own headline
  answer to target staleness - a transition's stored policy/value target
  was produced by whatever network existed *when it was collected*,
  which could be arbitrarily many gradient steps out of date by the time
  it's actually sampled for training. The reference runs whole separate
  background worker processes continuously re-`search()`-ing sampled
  trajectories with the network's current weights to refresh their
  stored targets; `_reanalyze` ports the same idea as a periodic in-loop
  pass instead (`reanalyze_freq`/`reanalyze_batch_size`, triggered from
  `learn()` the same num_envs-independent way `train_freq` is). A
  transition's value target is then `max(td_bootstrap_target,
  search_value)` - the reference's own `value_target: 'max'` mode -
  falling back transparently to the plain TD target for anything not yet
  reanalyzed.

Remaining simplifications vs. the paper (kept explicit here rather than
silent, matching this app's convention elsewhere - see e.g. `qmix.py`'s
own "Stability note"):
- No separate async self-play/reanalyze/train worker processes (Ray) -
  reanalyze above happens synchronously in-loop instead of on its own
  always-running process, so it directly costs wall-clock time rather
  than being "free" background work; tune `reanalyze_freq`/
  `reanalyze_batch_size` down if that overhead matters more than target
  freshness for a given env.
- Latent normalization is a plain `tanh` bound on `s`, not the paper's
  learned min-max rescaling - same spirit (keep the model's own latent
  space from exploding, so squared-error/cosine losses on it stay
  meaningful), simpler mechanism.
- No Dirichlet exploration noise mixed into root priors (the paper's
  standard MCTS-family exploration boost) - the Gumbel-Top-k sampling at
  the root already injects fresh per-step randomness into which actions
  even get a chance to be searched, which is Gumbel search's own built-in
  substitute for it.
"""
DEFAULT_HYPERPARAMS = {
    "learning_rate": 2e-4,
    "latent_dim": 64,
    "hidden_dim": 128,
    "proj_dim": 64,
    "buffer_size": 2_000,
    "batch_size": 64,
    "unroll_steps": 5,
    "td_steps": 5,
    "num_sampled_actions": 8,
    "num_simulations": 32,
    "num_top_actions": 8,
    "c_visit": 50.0,
    "c_scale": 0.1,
    "value_minmax_delta": 0.01,
    "value_support_size": 300,
    "gamma": 0.99,
    "value_loss_coef": 0.5,
    "policy_loss_coef": 1.0,
    "reward_loss_coef": 1.0,
    # 5.0, not 1.0 - matches the official EfficientZeroV2 repo's own
    # discrete/image-based (Atari) default config (ez/config/exp/atari.yaml:
    # consistency_coeff=5.0 vs value_loss_coeff=0.5, a 10x ratio; continuous
    # DMC configs use 2.0). Safe to weight this heavily specifically because
    # the projector/predictor's BatchNorm1d + stop-gradient (see _Projector/
    # _Predictor below) structurally prevents representation collapse
    # regardless of loss weight - that's the entire point of the SimSiam/
    # BYOL asymmetry, so a higher weight just makes the representation
    # converge faster/tighter rather than risking collapse.
    "consistency_loss_coef": 5.0,
    "continuous_prior_scale": 2.5,
    "policy_target_temperature": 1.0,
    "train_freq": 1,
    "train_steps_per_iter": 1,
    "learning_starts": 500,
    # Prioritized replay (Schaul et al., 2016) + reanalyze (target
    # staleness correction) - see `_EfficientZeroBuffer`/`_reanalyze`
    # docstrings. Values match the official EfficientZeroV2 repo's own
    # `ez/config/exp/atari.yaml` (`priority_prob_alpha`/`priority_prob_beta`
    # both 1.0, `min_prior` ~1e-6); `reanalyze_freq`/`reanalyze_batch_size`
    # are this port's own knobs for how often/how much to refresh (the
    # reference instead runs a whole separate always-on async worker
    # process for this - see module docstring).
    "priority_alpha": 1.0,
    "priority_beta": 1.0,
    "min_priority": 1e-6,
    "reanalyze_freq": 200,
    "reanalyze_batch_size": 64,
    "max_grad_norm": 5.0,
}

class _Representation(nn.Module):
    """`h(obs) -> s` - reuses the exact same image/vector auto-detecting
    feature extractor every world model family in this app shares
    (`ObsEncoder`), plus a small projection
    head down to the (much smaller) latent size the dynamics/prediction
    nets actually plan in."""

    def __init__(self, observation_space: gym.Space, latent_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.encoder = ObsEncoder(observation_space)
        self.head = nn.Sequential(
            nn.Linear(self.encoder.out_dim, hidden_dim), nn.ELU(), nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return torch.tanh(self.head(self.encoder(obs)))


class _Dynamics(nn.Module):
    """`g(s, a) -> (s', value_prefix)` - deterministic latent transition
    (no stochastic branching, unlike Dreamer's RSSM: a MuZero-style latent
    is meant to be planned through cheaply many times per real step, which
    a stochastic one would make far more expensive) plus the LSTM "value
    prefix" reward head described in the module docstring. The LSTM state
    must be reset (`initial_lstm_state`) at the start of every fresh
    unroll - one training unroll or one search rollout - since its output
    is a *cumulative* sum from wherever that reset happened, not a
    standalone per-step reward."""

    def __init__(self, latent_dim: int, action_dim: int, hidden_dim: int, support_size: int) -> None:
        super().__init__()
        self.hidden_dim = hidden_dim
        self.support_size = support_size
        self.state_encoder = nn.Linear(latent_dim, hidden_dim)
        self.action_encoder = nn.Linear(action_dim, hidden_dim)
        self.trunk = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.ELU())
        self.next_latent_head = nn.Linear(hidden_dim, latent_dim)
        self.lstm = nn.LSTMCell(hidden_dim, hidden_dim)
        self.value_prefix_head = nn.Linear(hidden_dim, 2 * support_size + 1)

    def initial_lstm_state(self, batch_size: int, device: str) -> tuple[torch.Tensor, torch.Tensor]:
        h = torch.zeros(batch_size, self.hidden_dim, device=device)
        c = torch.zeros(batch_size, self.hidden_dim, device=device)
        return h, c

    def forward(
        self, s: torch.Tensor, action: torch.Tensor, lstm_state: tuple[torch.Tensor, torch.Tensor],
    ) -> tuple[torch.Tensor, torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
        """Returns `value_prefix_logits` (categorical, see module docstring's
        "Categorical value/reward" bullet) - use `_logits_to_scalar` for the
        raw cumulative-reward estimate, or feed the logits straight into a
        cross-entropy loss against `_scalar_to_two_hot(cum_reward, ...)`."""
        x = torch.cat([self.state_encoder(s), self.action_encoder(action)], dim=-1)
        h = self.trunk(x)
        next_s = torch.tanh(self.next_latent_head(h))
        lstm_h, lstm_c = self.lstm(h, lstm_state)
        value_prefix_logits = self.value_prefix_head(lstm_h)
        return next_s, value_prefix_logits, (lstm_h, lstm_c)


class _Prediction(nn.Module):
    """`f(s) -> (policy, value)` - discrete actions get plain softmax
    logits (fed to the search as Gumbel-Top-k proposal scores); continuous
    actions get a diagonal Gaussian, exactly like `dreamer.py`'s `_Actor`
    (same tanh-squash-and-rescale-to-bounds convention)."""

    def __init__(
        self, latent_dim: int, action_space: gym.Space, hidden_dim: int, support_size: int,
    ) -> None:
        super().__init__()
        self.discrete = isinstance(action_space, gym.spaces.Discrete)
        self.support_size = support_size
        self.trunk = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ELU())
        self.value_head = nn.Linear(hidden_dim, 2 * support_size + 1)
        if self.discrete:
            self.action_dim = int(action_space.n)
            self.policy_head = nn.Linear(hidden_dim, self.action_dim)
        else:
            self.action_dim = int(np.prod(action_space.shape))
            self.mean_head = nn.Linear(hidden_dim, self.action_dim)
            self.log_std_head = nn.Linear(hidden_dim, self.action_dim)
            low = np.where(np.isfinite(action_space.low), action_space.low, -1.0).reshape(-1)
            high = np.where(np.isfinite(action_space.high), action_space.high, 1.0).reshape(-1)
            self.register_buffer("action_scale", torch.as_tensor((high - low) / 2.0, dtype=torch.float32))
            self.register_buffer("action_bias", torch.as_tensor((high + low) / 2.0, dtype=torch.float32))

    def value_logits(self, s: torch.Tensor) -> torch.Tensor:
        """Categorical value-head logits - see module docstring's
        "Categorical value/reward" bullet. Used as-is for the training
        cross-entropy loss; `value()` below converts to a raw scalar for
        every other caller (search bookkeeping, TD bootstrap target)."""
        return self.value_head(self.trunk(s))

    def value(self, s: torch.Tensor) -> torch.Tensor:
        return _logits_to_scalar(self.value_logits(s), self.support_size)

    def policy_logits(self, s: torch.Tensor) -> torch.Tensor:
        return self.policy_head(self.trunk(s))

    def policy_gaussian(self, s: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.trunk(s)
        mean = torch.tanh(self.mean_head(h)) * self.action_scale + self.action_bias
        log_std = self.log_std_head(h).clamp(-5.0, 1.0)
        return mean, log_std.exp()


class _Projector(nn.Module):
    """SimSiam-style projector `P1` (module docstring's consistency loss).
    `BatchNorm1d` on the hidden layer is not decorative - Chen & He, 2021's
    own ablation (Table 2c) shows stop-gradient *alone* still collapses in
    practice (every latent converging to ~the same vector, which trivially
    "matches" whatever the predictor outputs and costs nothing at all to
    reach); BN is what empirically keeps that from happening by forcing
    every batch's projected features to actually spread out (zero mean,
    unit variance *per feature, across the batch*) instead of letting the
    optimizer collapse them all together."""

    def __init__(self, latent_dim: int, proj_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, proj_dim), nn.BatchNorm1d(proj_dim), nn.ELU(), nn.Linear(proj_dim, proj_dim),
        )

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        return self.net(s)


class _Predictor(nn.Module):
    """SimSiam-style predictor `P2` - only applied on the *predicted*
    (dynamics) branch, never the stop-gradiented real-encoding branch
    (asymmetry), with the same anti-collapse `BatchNorm1d` as `_Projector`
    on its hidden layer - see that class's docstring."""

    def __init__(self, proj_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(proj_dim, proj_dim), nn.BatchNorm1d(proj_dim), nn.ELU(), nn.Linear(proj_dim, proj_dim),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


def _softmax(x: np.ndarray) -> np.ndarray:
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()


# ----------------------------------------------------------------------
# Categorical value/reward support - MuZero Appendix F "scaling and
# squashing" (`h`/`h^-1`), same closed-form transform DeepMind's own
# `rlax.signed_hyperbolic`/`signed_parabolic` use - see module docstring's
# "Categorical value/reward" bullet for *why* this exists (bounding the
# loss/gradient of the value & value-prefix heads regardless of how large
# one particular reward/value target happens to be).
# ----------------------------------------------------------------------
def _signed_hyperbolic(x: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    return torch.sign(x) * (torch.sqrt(torch.abs(x) + 1.0) - 1.0) + eps * x


def _signed_parabolic(x: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    z = torch.sqrt(1.0 + 4.0 * eps * (torch.abs(x) + 1.0 + eps)) - 1.0
    return torch.sign(x) * ((z / (2.0 * eps)) ** 2 - 1.0)


def _scalar_to_two_hot(x: torch.Tensor, support_size: int) -> torch.Tensor:
    """Raw scalar(s) `x` (any shape) -> two-hot distribution over
    `2 * support_size + 1` bins (new trailing dim), in the
    `_signed_hyperbolic`-transformed scale - the cross-entropy target for
    both the value and value-prefix heads."""
    num_bins = 2 * support_size + 1
    z = _signed_hyperbolic(x).clamp(-support_size, support_size)
    floor = z.floor()
    frac = (z - floor).unsqueeze(-1)
    floor_idx = (floor + support_size).long().clamp(0, num_bins - 1)
    ceil_idx = (floor_idx + 1).clamp(0, num_bins - 1)
    two_hot = torch.zeros(*x.shape, num_bins, device=x.device, dtype=x.dtype)
    two_hot.scatter_(-1, floor_idx.unsqueeze(-1), 1.0 - frac)
    two_hot.scatter_add_(-1, ceil_idx.unsqueeze(-1), frac)
    return two_hot


def _logits_to_scalar(logits: torch.Tensor, support_size: int) -> torch.Tensor:
    """Predicted per-bin logits -> raw scalar (expectation over bins in the
    transformed scale, then `_signed_parabolic` back to raw)."""
    probs = F.softmax(logits, dim=-1)
    support = torch.arange(-support_size, support_size + 1, device=logits.device, dtype=probs.dtype)
    return _signed_parabolic((probs * support).sum(-1))


class _MinMaxStats:
    """Running min/max of every backed-up value seen so far in one search
    tree, used to normalize Q-values into `[0, 1]` before they're compared
    against each other (Danihelka et al., 2022, Appendix A) - without this
    a tree whose rewards/values happen to live on a different scale than
    `c_visit`/`c_scale` were tuned for would search essentially randomly."""

    def __init__(self, delta: float) -> None:
        self.maximum = -float("inf")
        self.minimum = float("inf")
        self.delta = max(1e-6, delta)

    def update(self, value: float) -> None:
        self.maximum = max(self.maximum, value)
        self.minimum = min(self.minimum, value)

    def normalize(self, value: float) -> float:
        if self.maximum > self.minimum:
            value = min(max(value, self.minimum), self.maximum)
            value = (value - self.minimum) / max(self.maximum - self.minimum, self.delta)
        return min(max(value, 0.0), 1.0)


class _SearchNode:
    """One node of the Gumbel search tree (Danihelka et al., 2022) - either
    a root (one per lane being searched this real step) or an imagined
    node reached from its parent by taking one of the parent's candidate
    actions and stepping the dynamics net. Every candidate action slot
    gets its own (initially un-expanded) child node the instant its parent
    is expanded - "un-expanded child" and "child the tree hasn't
    visited/expanded yet" are the same thing here, exactly like the
    reference tree."""

    __slots__ = (
        "prior", "parent", "children", "visit_count", "value_sum", "value_prefix",
        "reset_value_prefix", "state", "lstm_state", "candidate_action", "selected_children_idx",
    )

    def __init__(self, prior: float, parent: "_SearchNode | None" = None) -> None:
        self.prior = prior
        self.parent = parent
        self.children: list[_SearchNode] = []
        self.visit_count = 0
        self.value_sum = 0.0
        self.value_prefix = 0.0
        self.reset_value_prefix = True
        self.state: torch.Tensor | None = None
        self.lstm_state: tuple[torch.Tensor, torch.Tensor] | None = None
        self.candidate_action: Any = None  # continuous only: the sampled action leading to this node
        self.selected_children_idx: list[int] = []  # root only: Sequential Halving survivors

    def expanded(self) -> bool:
        return len(self.children) > 0

    def value(self) -> float:
        return self.value_sum / self.visit_count if self.visit_count > 0 else 0.0

    def reward(self) -> float:
        """Real reward this node's edge contributed - value prefix is a
        *cumulative* sum since the last LSTM reset, so a node reached right
        after a reset reports its value prefix as-is, otherwise the delta
        against its parent's (Ye et al., 2021 value-prefix convention)."""
        if self.reset_value_prefix or self.parent is None:
            return self.value_prefix
        return self.value_prefix - self.parent.value_prefix

    def children_visit_sum(self) -> int:
        return sum(c.visit_count for c in self.children)

    def q_of_child(self, idx: int, discount: float) -> float:
        child = self.children[idx]
        return child.reward() + discount * child.value()

    def v_mix(self, discount: float) -> float:
        """Appendix D of Danihelka et al., 2022: blend this node's own raw
        value estimate with however many of its children are already
        expanded, weighted by how much of the node's total prior
        probability mass those expanded children cover - gives
        `completed_qs` something better than "value at the root" to use
        as a stand-in Q for children the tree hasn't visited at all yet."""
        priors = np.array([c.prior for c in self.children], dtype=np.float64)
        pi = _softmax(priors)
        pi_sum = 0.0
        pi_q_sum = 0.0
        for i, child in enumerate(self.children):
            if child.expanded():
                pi_sum += pi[i]
                pi_q_sum += pi[i] * self.q_of_child(i, discount)
        if pi_sum < 1e-6:
            return self.value()
        visit_sum = self.children_visit_sum()
        return (self.value() + visit_sum * pi_q_sum / pi_sum) / (1.0 + visit_sum)

    def completed_qs(self, discount: float, minmax: _MinMaxStats) -> np.ndarray:
        v_mix = self.v_mix(discount)
        out = np.empty(len(self.children), dtype=np.float64)
        for i, child in enumerate(self.children):
            raw = self.q_of_child(i, discount) if child.expanded() else v_mix
            out[i] = minmax.normalize(raw)
        return out

    def improved_policy(self, transformed_completed_qs: np.ndarray) -> np.ndarray:
        priors = np.array([c.prior for c in self.children], dtype=np.float64)
        return _softmax(priors + transformed_completed_qs)

    def expand(
        self, priors: np.ndarray, state: torch.Tensor, value_prefix: float,
        lstm_state: tuple[torch.Tensor, torch.Tensor], reset_value_prefix: bool,
        candidate_actions: list[Any] | None = None,
    ) -> None:
        self.state = state
        self.value_prefix = value_prefix
        self.lstm_state = lstm_state
        self.reset_value_prefix = reset_value_prefix
        self.children = [_SearchNode(float(p), self) for p in priors]
        if candidate_actions is not None:
            for child, action in zip(self.children, candidate_actions):
                child.candidate_action = action


def _transformed_completed_qs(node: _SearchNode, minmax: _MinMaxStats, discount: float, c_visit: float, c_scale: float) -> np.ndarray:
    completed = node.completed_qs(discount, minmax)
    max_visit = max((c.visit_count for c in node.children), default=0)
    return (c_visit + max_visit) * c_scale * completed


def _select_action(node: _SearchNode, minmax: _MinMaxStats, discount: float, c_visit: float, c_scale: float) -> int:
    """Root: equal-visit round robin over the current Sequential-Halving
    survivors (`selected_children_idx`) - ties broken towards whichever
    survivor is earliest in Gumbel-Top-k rank, matching the reference
    `do_equal_visit`. Non-root: the paper's deterministic selection rule -
    argmax of (improved policy - visit fraction) over *all* children, no
    Sequential Halving below the root."""
    if node.parent is None:
        best_idx, best_visits = -1, float("inf")
        for idx in node.selected_children_idx:
            visits = node.children[idx].visit_count
            if visits < best_visits:
                best_visits, best_idx = visits, idx
        return best_idx
    transformed = _transformed_completed_qs(node, minmax, discount, c_visit, c_scale)
    improved = node.improved_policy(transformed)
    visits = np.array([c.visit_count for c in node.children], dtype=np.float64)
    denom = 1.0 + node.children_visit_sum()
    scores = improved - visits / denom
    return int(np.argmax(scores))


def _sequential_halving(
    root: _SearchNode, gumbel: np.ndarray, minmax: _MinMaxStats, keep: int, discount: float, c_visit: float, c_scale: float,
) -> None:
    """Cut the root's currently-surviving candidate set in half (by score =
    Gumbel noise + prior + transformed completed Q), same formula the
    reference `sequential_halving` uses past its first phase - the first
    phase (Gumbel-Top-k over *all* actions) is instead folded into where
    `selected_children_idx` gets initialized (simulation 0 of `search()`
    below), which is exactly equivalent given the reference's own
    equal-visit round robin always exhausts the top `num_top_actions`
    candidates before any lower-ranked one ever gets a look."""
    selected = root.selected_children_idx
    if len(selected) <= 1:
        return
    priors = np.array([c.prior for c in root.children], dtype=np.float64)
    transformed = _transformed_completed_qs(root, minmax, discount, c_visit, c_scale)
    scores = np.array([gumbel[i] + priors[i] + transformed[i] for i in selected])
    order = np.argsort(-scores)
    keep = max(1, min(keep, len(selected)))
    root.selected_children_idx = [selected[i] for i in order[:keep]]


def _backpropagate(path: list[_SearchNode], leaf_value: float, minmax: _MinMaxStats, discount: float) -> None:
    value = leaf_value
    for node in reversed(path):
        node.value_sum += value
        node.visit_count += 1
        value = node.reward() + discount * value
        minmax.update(value)


class _HalvingSchedule:
    """Ports the reference `ready_for_next_gumble_phase`'s visit-budget
    schedule verbatim: phase 0 gets `floor(n / (log2(m) * m)) * m`
    simulations (always a multiple of `m = num_top_actions`, so the root's
    equal-visit round robin exhausts exactly the top-`m` Gumbel-Top-k
    candidates without ever spilling into a lower-ranked one - see
    `_sequential_halving`'s docstring), then each subsequent phase gets
    `floor(n / (log2(m) * current_m)) * current_m` more (or whatever's
    left, once `current_m <= 2`), halving `current_m` every time, until
    the budget of `n = num_simulations` simulations runs out."""

    def __init__(self, num_simulations: int, num_top_actions: int) -> None:
        self.n = max(1, num_simulations)
        self.m = max(2, num_top_actions)
        self.current_top = self.m
        self.next_cutoff = self._span(self.current_top, used=0)

    def _span(self, current_top: int, used: int) -> int:
        log2m = max(math.log2(self.m), 1e-9)
        if current_top > 2:
            span = math.floor(self.n / (log2m * current_top)) * current_top
        else:
            span = self.n - used
        return min(self.n, max(1, int(span)))

    def maybe_advance(self, simulation_idx: int) -> bool:
        ready = (simulation_idx + 1) >= self.next_cutoff
        if ready and self.current_top > 1:
            used = self.next_cutoff
            self.current_top = max(1, self.current_top // 2)
            self.next_cutoff = min(self.n, used + self._span(self.current_top, used))
        return ready


class _EfficientZeroBuffer:
    """Stores whole real episodes: `obs[t]`, `action[t]` (flat one-hot/raw
    - one-hot for a discrete action, the raw vector for a continuous one), `reward[t]`,
    `next_obs[t]` and `policy_target[t]` (this step's Gumbel/sampling
    search result at collection time - see module docstring). A dedicated
    buffer rather than a plain sequence replay: those have no slot for the
    search-derived policy target, which is this algorithm's actual
    training signal for the policy head (the cross-entropy/regression
    target in `_train_step`, not the raw played action).

    Two more per-transition arrays exist purely to port the official
    EfficientZeroV2 repo's own training recipe (`ez/agents/base.py`,
    `priority.use_priority: True`, `train.value_target: 'mixed'`) instead
    of the simpler "single TD target, uniform replay" scheme this file
    used before:
    - `priority[t]`: `|value_prediction - value_target|` from the last
      time this transition was trained on (reference's own `fresh_priority`
      - see `_train_step`), used for *prioritized* sampling below instead
      of uniform - new/never-trained transitions start at the buffer's
      running max priority so they get picked up promptly (standard PER
      convention, avoids fresh data starving under a stale-but-still-high
      initial priority estimate).
    - `search_value[t]`: root value from the *most recent* `search()` call
      over this transition's `obs[t]` - refreshed periodically by
      `EfficientZeroV2._reanalyze` using the network's *current*
      weights, not just whatever the (possibly much less trained) network
      produced at collection time. `_train_step` takes
      `max(td_bootstrap_target, search_value)` as the value target - the
      reference's `value_target: 'max'` mode - so a transition that's
      never been reanalyzed (`search_value` stays at its `-inf`-ish
      sentinel) transparently falls back to the plain TD target.

    `num_lanes > 1` (`training.num_envs > 1`, see module docstring) lets
    `add(..., lane=i)` accumulate `num_lanes` independent in-progress
    episodes concurrently without their interleaved transitions splicing
    into one another."""

    _NO_SEARCH_VALUE = -1e9

    def __init__(
        self, capacity_episodes: int, obs_shape: tuple[int, ...], action_dim: int, policy_target_dim: int,
        num_lanes: int = 1, priority_alpha: float = 1.0, priority_beta: float = 1.0, min_priority: float = 1e-6,
    ) -> None:
        self.capacity = max(1, capacity_episodes)
        self.obs_shape = obs_shape
        self.action_dim = action_dim
        self.policy_target_dim = policy_target_dim
        self._storage_dtype = np.float16 if max(action_dim, policy_target_dim) >= 512 else np.float32
        self.priority_alpha = max(0.0, priority_alpha)
        self.priority_beta = max(0.0, priority_beta)
        self.min_priority = max(1e-8, min_priority)
        self.episodes: list[dict[str, np.ndarray]] = []
        self._episode_alpha_sum: list[float] = []
        self._episode_outcomes: list[int | None] = []
        self._max_priority = 1.0
        self._cur: list[dict[str, list[Any]]] = [self._new_episode() for _ in range(max(1, num_lanes))]

    @staticmethod
    def _new_episode() -> dict[str, list[Any]]:
        return {"obs": [], "action": [], "reward": [], "next_obs": [], "policy_target": []}

    def add(
        self, obs: np.ndarray, action_flat: np.ndarray, reward: float, next_obs: np.ndarray,
        policy_target: np.ndarray, done: bool, lane: int = 0, episode_outcome: int | None = None,
    ) -> None:
        cur = self._cur[lane]
        cur["obs"].append(np.asarray(obs, dtype=self._storage_dtype))
        cur["action"].append(np.asarray(action_flat, dtype=self._storage_dtype))
        cur["reward"].append(float(reward))
        cur["next_obs"].append(np.asarray(next_obs, dtype=self._storage_dtype))
        cur["policy_target"].append(np.asarray(policy_target, dtype=self._storage_dtype))
        if done:
            self._flush_episode(lane, episode_outcome)

    def _flush_episode(self, lane: int = 0, episode_outcome: int | None = None) -> None:
        cur = self._cur[lane]
        if not cur["obs"]:
            return
        ep_len = len(cur["obs"])
        episode = {
            "obs": np.stack(cur["obs"]),
            "action": np.stack(cur["action"]),
            "reward": np.asarray(cur["reward"], dtype=np.float32),
            "next_obs": np.stack(cur["next_obs"]),
            "policy_target": np.stack(cur["policy_target"]),
            # New data starts at the running max priority (PER convention)
            # so it gets sampled/trained-on promptly rather than starved by
            # whatever priority scale earlier, already-trained data has.
            "priority": np.full(ep_len, self._max_priority, dtype=np.float64),
            "search_value": np.full(ep_len, self._NO_SEARCH_VALUE, dtype=np.float32),
        }
        self.episodes.append(episode)
        self._episode_alpha_sum.append(float(np.sum(episode["priority"] ** self.priority_alpha)))
        self._episode_outcomes.append(
            int(episode_outcome) if episode_outcome in (-1, 0, 1) else None,
        )
        if len(self.episodes) > self.capacity:
            evict = self_play_eviction_index(self._episode_outcomes, self.capacity)
            self.episodes.pop(evict)
            self._episode_alpha_sum.pop(evict)
            self._episode_outcomes.pop(evict)
        self._cur[lane] = self._new_episode()

    def __len__(self) -> int:
        return sum(len(ep["reward"]) for ep in self.episodes) + sum(len(cur["reward"]) for cur in self._cur)

    @property
    def num_episodes(self) -> int:
        return len(self.episodes)

    def sample(self, batch_size: int, unroll_steps: int, td_steps: int, gamma: float) -> dict[str, np.ndarray]:
        """Prioritized (episode, start index) per batch element - two-level
        proportional sampling (pick an episode weighted by its total
        `priority ** alpha`, then a start index within it weighted by its
        *own* `priority ** alpha`) rather than a single flat sum-tree over
        every transition in the buffer, which would cost `O(total
        transitions)` to rebuild on every priority update. This is the
        standard hierarchical approximation to flat proportional-priority
        sampling and is exact for the common case of near-uniform priority
        *within* an episode. `unroll_steps` contiguous real transitions
        from the sampled start (padded/masked past episode end), plus a
        `td_steps`-step (truncated at episode end) real-reward sum +
        bootstrap observation/discount/mask, and the stored `search_value`
        (see class docstring's `value_target: 'max'` note) for the value
        target; `episode_idx`/`timestep`/`is_weight` let `_train_step`
        write fresh priorities back after computing the loss (reference's
        own `fresh_priority`) and correct the loss for the sampling bias
        prioritization introduces (importance-sampling weights, Schaul et
        al., 2016)."""
        obs0 = np.zeros((batch_size, *self.obs_shape), dtype=np.float32)
        action = np.zeros((batch_size, unroll_steps, self.action_dim), dtype=np.float32)
        reward = np.zeros((batch_size, unroll_steps), dtype=np.float32)
        next_obs = np.zeros((batch_size, unroll_steps, *self.obs_shape), dtype=np.float32)
        policy_target = np.zeros((batch_size, unroll_steps, self.policy_target_dim), dtype=np.float32)
        mask = np.zeros((batch_size, unroll_steps), dtype=np.float32)
        td_reward = np.zeros((batch_size, unroll_steps), dtype=np.float32)
        td_obs = np.zeros((batch_size, unroll_steps, *self.obs_shape), dtype=np.float32)
        td_discount = np.zeros((batch_size, unroll_steps), dtype=np.float32)
        td_bootstrap_mask = np.zeros((batch_size, unroll_steps), dtype=np.float32)
        search_value = np.full((batch_size, unroll_steps), self._NO_SEARCH_VALUE, dtype=np.float32)
        episode_idx = np.zeros(batch_size, dtype=np.int64)
        timestep = np.zeros(batch_size, dtype=np.int64)
        sample_prob = np.zeros(batch_size, dtype=np.float64)

        n_episodes = len(self.episodes)
        alpha_sums = np.asarray(self._episode_alpha_sum, dtype=np.float64)
        total_alpha_sum = float(alpha_sums.sum())
        if total_alpha_sum > 0:
            ep_probs = alpha_sums / total_alpha_sum
        else:
            ep_probs = np.full(n_episodes, 1.0 / n_episodes)
        ep_choices = np.random.choice(n_episodes, size=batch_size, p=ep_probs)
        n_total_transitions = max(1, len(self))

        for b in range(batch_size):
            ep_i = int(ep_choices[b])
            ep = self.episodes[ep_i]
            t_len = len(ep["reward"])
            local_weight = ep["priority"] ** self.priority_alpha
            local_sum = float(local_weight.sum())
            if local_sum > 0:
                local_probs = local_weight / local_sum
                start = int(np.random.choice(t_len, p=local_probs))
                local_prob = float(local_probs[start])
            else:
                start = int(np.random.randint(t_len))
                local_prob = 1.0 / t_len
            episode_idx[b] = ep_i
            timestep[b] = start
            sample_prob[b] = max(ep_probs[ep_i] * local_prob, 1e-12)

            obs0[b] = ep["obs"][start]
            for k in range(unroll_steps):
                idx = start + k
                if idx >= t_len:
                    continue
                mask[b, k] = 1.0
                action[b, k] = ep["action"][idx]
                reward[b, k] = ep["reward"][idx]
                next_obs[b, k] = ep["next_obs"][idx]
                policy_target[b, k] = ep["policy_target"][idx]
                search_value[b, k] = ep["search_value"][idx]

                n = min(td_steps, t_len - idx)
                td_sum, discount = 0.0, 1.0
                for j in range(n):
                    td_sum += discount * float(ep["reward"][idx + j])
                    discount *= gamma
                td_reward[b, k] = td_sum
                td_discount[b, k] = discount
                landing_idx = idx + n - 1  # obs reached after n real actions from idx
                td_obs[b, k] = ep["next_obs"][landing_idx]
                td_bootstrap_mask[b, k] = 0.0 if landing_idx == t_len - 1 else 1.0

        # Importance-sampling correction (Schaul et al., 2016, eq. 5):
        # w_i = (1 / (N * P(i))) ** beta, normalized by the batch's own max
        # so the largest weight in any given batch is always 1 (keeps the
        # overall loss scale stable regardless of how skewed priorities
        # currently are).
        is_weight = (1.0 / (n_total_transitions * sample_prob)) ** self.priority_beta
        is_weight = is_weight / max(float(is_weight.max()), 1e-12)

        return {
            "obs0": obs0, "action": action, "reward": reward, "next_obs": next_obs,
            "policy_target": policy_target, "mask": mask, "td_reward": td_reward,
            "td_obs": td_obs, "td_discount": td_discount, "td_bootstrap_mask": td_bootstrap_mask,
            "search_value": search_value, "episode_idx": episode_idx, "timestep": timestep,
            "is_weight": is_weight.astype(np.float32),
        }

    def update_priorities(self, episode_idx: np.ndarray, timestep: np.ndarray, values: np.ndarray) -> None:
        """Write fresh `|value_prediction - value_target|` priorities back
        for exactly the transitions a `sample()` call just returned (safe
        to call with plain `self.episodes` indices: nothing in `learn()`
        ever calls `add()`/evicts an episode between a `sample()` and its
        matching `_train_step`'s priority write-back, both run fully
        synchronously inside one `_train_step()` call)."""
        values = np.maximum(np.asarray(values, dtype=np.float64), self.min_priority)
        touched = set()
        for ep_i, t, v in zip(episode_idx.tolist(), timestep.tolist(), values.tolist()):
            if not (0 <= ep_i < len(self.episodes)):
                continue
            self.episodes[ep_i]["priority"][t] = v
            touched.add(ep_i)
        self._max_priority = max(self._max_priority, float(values.max()) if len(values) else self._max_priority)
        for ep_i in touched:
            ep = self.episodes[ep_i]
            self._episode_alpha_sum[ep_i] = float(np.sum(ep["priority"] ** self.priority_alpha))

    def sample_for_reanalyze(self, n: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """`n` uniformly-random *distinct-ish* `(episode_idx, timestep)`
        pairs (plain uniform, unlike `sample()`'s priority weighting -
        reanalyze's job is to broadly refresh stale targets across the
        whole buffer, not concentrate on whatever's already
        high-priority) plus their `obs[timestep]`, for
        `EfficientZeroV2._reanalyze` to re-`search()` with the
        network's current weights."""
        n_episodes = len(self.episodes)
        n = min(n, sum(len(ep["reward"]) for ep in self.episodes))
        if n <= 0:
            empty_obs = np.zeros((0, *self.obs_shape), dtype=np.float32)
            return np.zeros(0, dtype=np.int64), np.zeros(0, dtype=np.int64), empty_obs
        lengths = np.asarray([len(ep["reward"]) for ep in self.episodes], dtype=np.float64)
        ep_choices = np.random.choice(n_episodes, size=n, p=lengths / lengths.sum())
        episode_idx = np.zeros(n, dtype=np.int64)
        timestep = np.zeros(n, dtype=np.int64)
        obs_out = np.zeros((n, *self.obs_shape), dtype=np.float32)
        for i, ep_i in enumerate(ep_choices.tolist()):
            ep = self.episodes[ep_i]
            t = int(np.random.randint(len(ep["reward"])))
            episode_idx[i] = ep_i
            timestep[i] = t
            obs_out[i] = ep["obs"][t]
        return episode_idx, timestep, obs_out

    def update_reanalyzed_targets(
        self, episode_idx: np.ndarray, timestep: np.ndarray, policy_targets: np.ndarray, search_values: np.ndarray,
    ) -> None:
        """Overwrite the stored (possibly very stale - collected who knows
        how many gradient steps ago) `policy_target`/`search_value` for
        exactly the transitions `sample_for_reanalyze` just returned, with
        fresh ones from a `search()` call using the network's *current*
        weights - the actual point of "reanalyze": target staleness
        correction, ported from the reference's own background reanalyze
        workers (`ez/worker/`) as a periodic in-loop pass instead of a
        separate async process (see `EfficientZeroV2._reanalyze`)."""
        for i, (ep_i, t) in enumerate(zip(episode_idx.tolist(), timestep.tolist())):
            if not (0 <= ep_i < len(self.episodes)):
                continue
            ep = self.episodes[ep_i]
            if t >= len(ep["reward"]):
                continue
            ep["policy_target"][t] = policy_targets[i]
            ep["search_value"][t] = search_values[i]


class EfficientZeroV2:
    """EfficientZero V2 для дискретных и непрерывных действий.

    Это перенос `EfficientZeroV2` без плагина студии: тот же Gumbel search,
    value prefix, categorical value/reward, consistency loss, prioritized replay
    и reanalyze. Для шахмат `two_player=True`, бэкап идёт с дисконтом `-gamma`.
    """

    def __init__(self, env: gym.Env, hyperparams: dict[str, Any], seed: int | None, device: str) -> None:
        self.env = env
        self.hyperparams = dict(hyperparams)
        self.seed = seed
        self.device = device
        if seed is not None:
            torch.manual_seed(seed)
            np.random.seed(seed)
        self._obs_space = venv_obs_space(env)
        self._action_space = venv_action_space(env)
        self.discrete = isinstance(self._action_space, gym.spaces.Discrete)
        self.n_actions = int(self._action_space.n) if self.discrete else None

        latent_dim = int(hyperparams.get("latent_dim", 64))
        hidden_dim = int(hyperparams.get("hidden_dim", 128))
        proj_dim = int(hyperparams.get("proj_dim", 64))
        self.action_dim = obs_flat_dim(self._action_space)

        self.support_size = max(1, int(hyperparams.get("value_support_size", 300)))
        self.representation = _Representation(self._obs_space, latent_dim, hidden_dim).to(device)
        self.dynamics = _Dynamics(latent_dim, self.action_dim, hidden_dim, self.support_size).to(device)
        self.prediction = _Prediction(latent_dim, self._action_space, hidden_dim, self.support_size).to(device)
        self.projector = _Projector(latent_dim, proj_dim).to(device)
        self.predictor = _Predictor(proj_dim).to(device)
        self._params = (
            list(self.representation.parameters()) + list(self.dynamics.parameters())
            + list(self.prediction.parameters()) + list(self.projector.parameters())
            + list(self.predictor.parameters())
        )
        self.optimizer = torch.optim.Adam(self._params, lr=float(hyperparams.get("learning_rate", 2e-4)))

        self.gamma = float(hyperparams.get("gamma", 0.99))
        attach_two_player(self, env)
        self.batch_size = int(hyperparams.get("batch_size", 64))
        self.unroll_steps = max(1, int(hyperparams.get("unroll_steps", 5)))
        self.td_steps = max(1, int(hyperparams.get("td_steps", 5)))
        self.num_sampled_actions = max(2, int(hyperparams.get("num_sampled_actions", 8)))
        self.num_simulations = max(2, int(hyperparams.get("num_simulations", 32)))
        self.num_top_actions = max(2, int(hyperparams.get("num_top_actions", 8)))
        self.c_visit = float(hyperparams.get("c_visit", 50.0))
        self.c_scale = float(hyperparams.get("c_scale", 0.1))
        self.value_minmax_delta = float(hyperparams.get("value_minmax_delta", 0.01))
        self.value_loss_coef = float(hyperparams.get("value_loss_coef", 0.5))
        self.policy_loss_coef = float(hyperparams.get("policy_loss_coef", 1.0))
        self.reward_loss_coef = float(hyperparams.get("reward_loss_coef", 1.0))
        self.consistency_loss_coef = float(hyperparams.get("consistency_loss_coef", 1.0))
        self.continuous_prior_scale = float(hyperparams.get("continuous_prior_scale", 2.5))
        self.policy_target_temperature = max(1e-6, float(hyperparams.get("policy_target_temperature", 1.0)))
        self.train_freq = max(1, int(hyperparams.get("train_freq", 1)))
        self.train_steps_per_iter = max(1, int(hyperparams.get("train_steps_per_iter", 1)))
        self.learning_starts = int(hyperparams.get("learning_starts", 500))
        self.max_grad_norm = float(hyperparams.get("max_grad_norm", 5.0))

        # Prioritized replay + reanalyze hyperparams - port of the official
        # EfficientZeroV2 repo's own default training recipe (`priority.*`/
        # `train.reanalyze_ratio`/`train.value_target: 'mixed'` in
        # `ez/config/exp/atari.yaml`), see `_EfficientZeroBuffer`'s and
        # `_reanalyze`'s docstrings for what each actually does.
        self.priority_alpha = max(0.0, float(hyperparams.get("priority_alpha", 1.0)))
        self.priority_beta = max(0.0, float(hyperparams.get("priority_beta", 1.0)))
        self.min_priority = max(1e-8, float(hyperparams.get("min_priority", 1e-6)))
        self.reanalyze_freq = max(1, int(hyperparams.get("reanalyze_freq", 200)))
        self.reanalyze_batch_size = max(0, int(hyperparams.get("reanalyze_batch_size", 64)))

        sample_obs_arr = obs_to_array(self._obs_space.sample(), self._obs_space)
        policy_target_dim = self.n_actions if self.discrete else self.action_dim
        requested_buffer_capacity = int(hyperparams.get("buffer_size", 2_000))
        effective_buffer_capacity = self_play_replay_capacity(
            requested_buffer_capacity,
            num_envs_of(env),
            self.two_player,
        )
        self.buffer = _EfficientZeroBuffer(
            capacity_episodes=effective_buffer_capacity,
            obs_shape=sample_obs_arr.shape, action_dim=self.action_dim, policy_target_dim=policy_target_dim,
            num_lanes=num_envs_of(env),
            priority_alpha=self.priority_alpha, priority_beta=self.priority_beta, min_priority=self.min_priority,
        )
        self._last_metrics: dict[str, float] = {
            "replay_capacity_episodes": float(effective_buffer_capacity),
            "replay_capacity_requested": float(requested_buffer_capacity),
        }

    # ------------------------------------------------------------------
    # Search - a genuine batched Gumbel search tree, see module docstring.
    # ------------------------------------------------------------------
    def _sample_continuous_candidates(self, mean: torch.Tensor, std: torch.Tensor) -> tuple[list[np.ndarray], np.ndarray]:
        """K = `num_sampled_actions` candidate child actions for one node,
        sampled from *that node's own* predicted Gaussian policy - `k1`
        "on policy" (its own std) plus `k2` "wide" (inflated std,
        `continuous_prior_scale`x), the paper's `A_{S1}`/`A_{S2}` split.
        Each candidate's prior is the (unnormalized) log-density of the
        on-policy Gaussian at that action - a consistent proposal-density
        prior for every slot regardless of which of the two samplers
        actually drew it, fed into the exact same Gumbel-Top-k/Sequential
        Halving/improved-policy formulas the discrete case uses."""
        mean_np = mean.squeeze(0).detach().cpu().numpy()
        std_np = std.squeeze(0).detach().cpu().numpy()
        low, high = self._action_space.low, self._action_space.high
        k1 = max(1, self.num_sampled_actions // 2)
        k2 = max(1, self.num_sampled_actions - k1)
        cand_policy = np.random.normal(mean_np, std_np, size=(k1, self.action_dim))
        cand_wide = np.random.normal(mean_np, std_np * self.continuous_prior_scale, size=(k2, self.action_dim))
        candidates = np.clip(np.concatenate([cand_policy, cand_wide], axis=0), low, high).astype(np.float32)
        var = np.clip(std_np, 1e-6, None) ** 2
        log_probs = -0.5 * (((candidates - mean_np) ** 2) / var + np.log(2 * np.pi * var)).sum(axis=-1)
        return list(candidates), log_probs.astype(np.float64)

    def search(
        self,
        obs_batch: np.ndarray,
        deterministic: np.ndarray | None = None,
        action_masks: np.ndarray | None = None,
    ) -> list[dict[str, Any]]:
        """Batched Gumbel search - `obs_batch`: `(B, *obs_shape)`. ALL B
        lanes' trees are searched together: one shared root
        representation/prediction pass, then `num_simulations` rounds
        where every lane descends its OWN tree by exactly one edge and all
        B lanes' chosen leaf expansions are computed in one batched
        dynamics/prediction forward pass (see module docstring). Returns a
        length-B list of `{"env_action", "policy_target", "value_target"}`.
        `deterministic`: `None`/scalar bool applies to every lane, or a
        length-B bool array for per-lane control (`predict()` vs `learn()`)."""
        batch_size = obs_batch.shape[0]
        det = np.zeros(batch_size, dtype=bool) if deterministic is None else np.broadcast_to(deterministic, (batch_size,))
        obs_t = torch.as_tensor(obs_batch, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            s0 = self.representation(obs_t)
            root_value = self.prediction.value(s0)
            if self.discrete:
                root_logits = self.prediction.policy_logits(s0)
            else:
                root_mean, root_std = self.prediction.policy_gaussian(s0)

        n_slots = self.n_actions if self.discrete else self.num_sampled_actions
        num_top_actions = max(2, min(self.num_top_actions, n_slots))
        discount = self.search_discount
        interior_k = interior_max_children(int(self.n_actions or 0)) if self.discrete else None
        roots: list[_SearchNode] = []
        for i in range(batch_size):
            root = _SearchNode(prior=1.0)
            lstm0 = self.dynamics.initial_lstm_state(1, self.device)
            if self.discrete:
                priors = root_logits[i].detach().cpu().numpy().astype(np.float64)
                mask = None if action_masks is None else action_masks[i]
                if mask is not None:
                    child_priors, action_ids = discrete_expand_slots(
                        priors, action_mask=mask, max_children=None, as_logits=True,
                    )
                    root.expand(child_priors, s0[i], 0.0, lstm0, True, candidate_actions=action_ids)
                else:
                    root.expand(priors, s0[i], 0.0, lstm0, True)
            else:
                candidates, priors = self._sample_continuous_candidates(root_mean[i : i + 1], root_std[i : i + 1])
                root.expand(priors, s0[i], 0.0, lstm0, True, candidate_actions=candidates)
            root.visit_count = 1
            root.value_sum = float(root_value[i].item())
            roots.append(root)

        minmax_list = [_MinMaxStats(self.value_minmax_delta) for _ in range(batch_size)]
        gumbel = [
            np.random.gumbel(size=len(root.children)) * self.policy_target_temperature
            for root in roots
        ]
        schedule = _HalvingSchedule(self.num_simulations, num_top_actions)
        lstm_horizon = self.unroll_steps

        for sim_idx in range(self.num_simulations):
            leaf_nodes: list[_SearchNode] = []
            parent_states, parent_lstm_h, parent_lstm_c = [], [], []
            chosen_actions: list[Any] = []
            search_paths: list[list[_SearchNode]] = []
            for lane in range(batch_size):
                root = roots[lane]
                if sim_idx == 0:
                    scores = gumbel[lane] + np.array([c.prior for c in root.children])
                    keep = min(num_top_actions, len(root.children))
                    root.selected_children_idx = list(np.argsort(-scores)[:keep])
                node = root
                path = [node]
                while node.expanded():
                    idx = _select_action(node, minmax_list[lane], discount, self.c_visit, self.c_scale)
                    node = node.children[idx]
                    path.append(node)
                parent = path[-2]
                leaf_nodes.append(node)
                parent_states.append(parent.state)
                parent_lstm_h.append(parent.lstm_state[0])
                parent_lstm_c.append(parent.lstm_state[1])
                chosen_actions.append(
                    node.candidate_action if not self.discrete else chosen_discrete_action(node, parent)
                )
                search_paths.append(path)

            parent_states_t = torch.stack(parent_states, dim=0)
            lstm_state = (torch.cat(parent_lstm_h, dim=0), torch.cat(parent_lstm_c, dim=0))
            if self.discrete:
                idx_t = torch.as_tensor(chosen_actions, dtype=torch.long, device=self.device)
                action_t = F.one_hot(idx_t, num_classes=self.n_actions).float()
            else:
                action_t = torch.as_tensor(np.stack(chosen_actions), dtype=torch.float32, device=self.device)
            with torch.no_grad():
                next_s, value_prefix_logits, next_lstm_state = self.dynamics(parent_states_t, action_t, lstm_state)
                value_prefix_scalar = _logits_to_scalar(value_prefix_logits, self.support_size)
                next_value = self.prediction.value(next_s)
                if self.discrete:
                    next_logits = self.prediction.policy_logits(next_s)
                else:
                    next_mean, next_std = self.prediction.policy_gaussian(next_s)

            search_lens = np.array([len(p) for p in search_paths])
            reset_mask = (search_lens % lstm_horizon == 0)
            for lane in range(batch_size):
                leaf = leaf_nodes[lane]
                lh, lc = next_lstm_state[0][lane : lane + 1], next_lstm_state[1][lane : lane + 1]
                if reset_mask[lane]:
                    lh, lc = torch.zeros_like(lh), torch.zeros_like(lc)
                vp = float(value_prefix_scalar[lane].item())
                if self.discrete:
                    priors = next_logits[lane].detach().cpu().numpy().astype(np.float64)
                    if interior_k is not None:
                        child_priors, action_ids = discrete_expand_slots(
                            priors, max_children=interior_k, as_logits=True,
                        )
                        leaf.expand(
                            child_priors, next_s[lane], vp, (lh, lc), bool(reset_mask[lane]),
                            candidate_actions=action_ids,
                        )
                    else:
                        leaf.expand(priors, next_s[lane], vp, (lh, lc), bool(reset_mask[lane]))
                else:
                    candidates, priors = self._sample_continuous_candidates(
                        next_mean[lane : lane + 1], next_std[lane : lane + 1],
                    )
                    leaf.expand(priors, next_s[lane], vp, (lh, lc), bool(reset_mask[lane]), candidate_actions=candidates)
                _backpropagate(search_paths[lane], float(next_value[lane].item()), minmax_list[lane], discount)

            if schedule.maybe_advance(sim_idx):
                for lane in range(batch_size):
                    _sequential_halving(
                        roots[lane], gumbel[lane], minmax_list[lane], schedule.current_top, discount, self.c_visit, self.c_scale,
                    )

        results: list[dict[str, Any]] = []
        for lane in range(batch_size):
            root = roots[lane]
            transformed = _transformed_completed_qs(root, minmax_list[lane], discount, self.c_visit, self.c_scale)
            improved = root.improved_policy(transformed)
            best_idx = int(root.selected_children_idx[0]) if root.selected_children_idx else int(np.argmax(improved))
            if self.discrete:
                policy_target = discrete_policy_target(root, improved, int(self.n_actions))
                if det[lane]:
                    env_action: Any = int(np.argmax(policy_target))
                else:
                    probs = policy_target / max(float(policy_target.sum()), 1e-8)
                    env_action = int(np.random.choice(len(probs), p=probs))
            else:
                candidates_arr = np.stack([c.candidate_action for c in root.children])
                policy_target = (improved[:, None] * candidates_arr).sum(axis=0).astype(np.float32)
                env_action = candidates_arr[best_idx].astype(np.float32)
            results.append({"env_action": env_action, "policy_target": policy_target, "value_target": root.value()})
        return results

    # ------------------------------------------------------------------
    # Reanalyze - port of the reference's own always-on background
    # reanalyze workers (`ez/worker/`), see `_EfficientZeroBuffer`'s class
    # docstring for what gets refreshed and why.
    # ------------------------------------------------------------------
    def _reanalyze(self) -> dict[str, float] | None:
        """Sample `reanalyze_batch_size` stored transitions (uniformly
        across the whole buffer, see `sample_for_reanalyze`) and refresh
        their `policy_target`/`search_value` with a fresh `search()` call
        using the network's *current* weights - one batched search, same
        cost as picking `reanalyze_batch_size` real actions at once. The
        reference runs this continuously in a separate process so it never
        competes with training/collection for time; ported here as a
        periodic in-loop pass instead (see `learn()`'s trigger, mirrors
        `train_freq`'s own num_envs-independent boundary counting) - the
        price of not needing a second process, at the cost of adding
        `reanalyze_batch_size`-worth of search compute every
        `reanalyze_freq` real steps."""
        # Historical legal masks are not part of this legacy replay
        # schema. Reanalysis without one poisons board-game targets; trying
        # to infer it from policy support also drops legal-but-unvisited
        # actions. Keep valid collection targets instead.
        if self.two_player:
            return None
        if self.reanalyze_batch_size <= 0 or self.buffer.num_episodes == 0:
            return None
        episode_idx, timestep, obs_batch = self.buffer.sample_for_reanalyze(self.reanalyze_batch_size)
        if obs_batch.shape[0] == 0:
            return None
        results = self.search(
            obs_batch,
            deterministic=np.zeros(obs_batch.shape[0], dtype=bool),
        )
        policy_targets = np.stack([r["policy_target"] for r in results])
        search_values = np.asarray([r["value_target"] for r in results], dtype=np.float32)
        self.buffer.update_reanalyzed_targets(episode_idx, timestep, policy_targets, search_values)
        return {"reanalyze_mean_search_value": float(search_values.mean())}

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------
    def _train_step(self) -> dict[str, float]:
        batch = self.buffer.sample(self.batch_size, self.unroll_steps, self.td_steps, self.search_discount)
        obs0 = torch.as_tensor(batch["obs0"], dtype=torch.float32, device=self.device)
        action = torch.as_tensor(batch["action"], dtype=torch.float32, device=self.device)
        reward = torch.as_tensor(batch["reward"], dtype=torch.float32, device=self.device)
        next_obs = torch.as_tensor(batch["next_obs"], dtype=torch.float32, device=self.device)
        policy_target = torch.as_tensor(batch["policy_target"], dtype=torch.float32, device=self.device)
        mask = torch.as_tensor(batch["mask"], dtype=torch.float32, device=self.device)
        td_reward = torch.as_tensor(batch["td_reward"], dtype=torch.float32, device=self.device)
        td_obs = torch.as_tensor(batch["td_obs"], dtype=torch.float32, device=self.device)
        td_discount = torch.as_tensor(batch["td_discount"], dtype=torch.float32, device=self.device)
        td_bootstrap_mask = torch.as_tensor(batch["td_bootstrap_mask"], dtype=torch.float32, device=self.device)
        search_value = torch.as_tensor(batch["search_value"], dtype=torch.float32, device=self.device)
        is_weight = torch.as_tensor(batch["is_weight"], dtype=torch.float32, device=self.device)
        cum_reward = torch.cumsum(reward, dim=1)

        batch_n = obs0.shape[0]
        s = self.representation(obs0)
        lstm_state = self.dynamics.initial_lstm_state(batch_n, self.device)
        reward_loss = torch.zeros((), device=self.device)
        value_loss = torch.zeros((), device=self.device)
        policy_loss = torch.zeros((), device=self.device)
        consistency_loss = torch.zeros((), device=self.device)
        first_step_value_pred: torch.Tensor | None = None
        first_step_value_target: torch.Tensor | None = None

        # Representation only ever runs on the *real* initial observation
        # (`obs0`) above - every subsequent `s` in this loop is the
        # dynamics net's own predicted latent, exactly like MuZero's own
        # training unroll: the model is trained to be accurate enough to
        # unroll through by itself, real observations only ever show up
        # again as loss *targets* (`next_obs`, `td_obs` below), never fed
        # back in to keep the rollout "on track".
        for k in range(self.unroll_steps):
            m = mask[:, k]
            # IS-weight multiplies in but the mean is still over the
            # (masked) *batch count*, not the weight sum - dividing by the
            # weight sum instead would exactly cancel out the bias
            # correction IS-weighting exists to apply (Schaul et al.,
            # 2016), same "(weights * loss).mean()" convention the
            # reference `update_weights` itself uses.
            wm = m * is_weight
            denom = m.sum().clamp_min(1.0)
            next_s, value_prefix_logits, lstm_state = self.dynamics(s, action[:, k], lstm_state)

            # Cross-entropy against a two-hot target, not scalar MSE - see
            # module docstring's "Categorical value/reward" bullet: bounds
            # the loss/gradient regardless of how large `cum_reward`/
            # `value_target` happens to be in this particular minibatch.
            reward_two_hot = _scalar_to_two_hot(cum_reward[:, k], self.support_size)
            r_loss = -(reward_two_hot * F.log_softmax(value_prefix_logits, dim=-1)).sum(-1)
            reward_loss = reward_loss + (r_loss * wm).sum() / denom

            value_pred_logits = self.prediction.value_logits(s)
            with torch.no_grad():
                bootstrap_value = self.prediction.value(self.representation(td_obs[:, k])) * td_bootstrap_mask[:, k]
                td_target = td_reward[:, k] + td_discount[:, k] * bootstrap_value
                # `value_target: 'max'` (reference's own mode, see
                # `_EfficientZeroBuffer`'s class docstring): transitions
                # `_reanalyze` has refreshed carry a real `search_value`
                # (the network's *current*-weights root value estimate,
                # generally more accurate/less stale than a plain n-step TD
                # bootstrap off however-old the value net was when this
                # transition was first collected) - take whichever target
                # is larger. Never-reanalyzed transitions keep their
                # `_NO_SEARCH_VALUE` sentinel, so this is a no-op fallback
                # to the plain TD target for them.
                value_target = torch.maximum(td_target, search_value[:, k])
            value_two_hot = _scalar_to_two_hot(value_target, self.support_size)
            v_loss = -(value_two_hot * F.log_softmax(value_pred_logits, dim=-1)).sum(-1)
            value_loss = value_loss + (v_loss * wm).sum() / denom
            if k == 0:
                with torch.no_grad():
                    first_step_value_pred = _logits_to_scalar(value_pred_logits, self.support_size).detach()
                    first_step_value_target = value_target.detach()

            if self.discrete:
                logp = F.log_softmax(self.prediction.policy_logits(s), dim=-1)
                p_loss = -(policy_target[:, k] * logp).sum(-1)
            else:
                mean, _std = self.prediction.policy_gaussian(s)
                p_loss = F.mse_loss(mean, policy_target[:, k], reduction="none").sum(-1)
            policy_loss = policy_loss + (p_loss * wm).sum() / denom

            with torch.no_grad():
                true_next_s = self.representation(next_obs[:, k])
            p_true = F.normalize(self.projector(true_next_s), dim=-1).detach()
            p_pred = F.normalize(self.predictor(self.projector(next_s)), dim=-1)
            c_loss = -(p_true * p_pred).sum(-1)
            consistency_loss = consistency_loss + (c_loss * wm).sum() / denom

            s = next_s

        n = float(self.unroll_steps)
        reward_loss, value_loss, policy_loss, consistency_loss = (
            reward_loss / n, value_loss / n, policy_loss / n, consistency_loss / n,
        )
        total_loss = (
            self.reward_loss_coef * reward_loss + self.value_loss_coef * value_loss
            + self.policy_loss_coef * policy_loss + self.consistency_loss_coef * consistency_loss
        )
        self.optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self._params, self.max_grad_norm)
        self.optimizer.step()

        # Fresh per-transition priorities from this step's own first-unroll-
        # position |prediction - target| (reference's `fresh_priority`) -
        # feeds back into `_EfficientZeroBuffer`'s prioritized sampling for
        # whichever *specific* (episode, timestep) pairs this batch just
        # trained on.
        assert first_step_value_pred is not None and first_step_value_target is not None
        fresh_priority = (first_step_value_pred - first_step_value_target).abs().cpu().numpy()
        self.buffer.update_priorities(batch["episode_idx"], batch["timestep"], fresh_priority)

        return {
            "reward_loss": float(reward_loss.item()),
            "value_loss": float(value_loss.item()),
            "policy_loss": float(policy_loss.item()),
            "consistency_loss": float(consistency_loss.item()),
            "mean_priority": float(fresh_priority.mean()) if fresh_priority.size else 0.0,
            "mean_is_weight": float(is_weight.mean().item()),
            "total_loss": float(total_loss.item()),
        }

    # ------------------------------------------------------------------
    # цикл обучения и предсказание хода
    # ------------------------------------------------------------------
    def learn(self, total_timesteps: int, callback: Any) -> None:
        n_envs = num_envs_of(self.env)
        obs_list, infos = vec_reset(self.env, seed=self.seed, return_info=True)
        maybe_enable_two_player_from_infos(self, infos)
        self.buffer.capacity = self_play_replay_capacity(
            self.buffer.capacity, n_envs, self.two_player,
        )
        obs_arr = obs_batch_to_array(obs_list, self._obs_space)
        action_masks = masks_from_infos(infos, int(self.n_actions or 0)) if self.discrete else None
        ep_reward = np.zeros(n_envs, dtype=np.float64)
        ep_length = np.zeros(n_envs, dtype=np.int64)
        num_timesteps = 0

        while num_timesteps < total_timesteps:
            env_actions: list[Any] = []
            action_flats: list[np.ndarray] = []
            policy_targets: list[np.ndarray] = []
            if num_timesteps < self.learning_starts:
                for lane in range(n_envs):
                    mask = None if action_masks is None else action_masks[lane]
                    if self.discrete:
                        raw_action = sample_masked_action(self._action_space, mask)
                        policy_target = uniform_legal_policy(int(self.n_actions), mask)
                    else:
                        raw_action = self._action_space.sample()
                        policy_target = np.asarray(raw_action, dtype=np.float32).reshape(-1)
                    env_action = action_to_env(raw_action, self._action_space)
                    env_actions.append(env_action)
                    action_flats.append(obs_to_array(env_action, self._action_space))
                    policy_targets.append(policy_target)
            else:
                # One shared, batched `search()` call plans all `n_envs`
                # lanes' trees together this real step (see module
                # docstring) - not a Python loop over independent searches.
                results = self.search(obs_arr, deterministic=None, action_masks=action_masks)
                for result in results:
                    env_action = action_to_env(result["env_action"], self._action_space)
                    env_actions.append(env_action)
                    action_flats.append(obs_to_array(env_action, self._action_space))
                    policy_targets.append(result["policy_target"])

            next_obs_list, rewards, terminated, truncated, infos = vec_step(self.env, env_actions)
            maybe_enable_two_player_from_infos(self, infos)
            dones = terminated | truncated
            record_self_play_outcomes(self, infos, terminated, truncated)
            next_obs_arr = obs_batch_to_array(next_obs_list, self._obs_space)
            action_masks = masks_from_infos(infos, int(self.n_actions or 0)) if self.discrete else None
            transition_next_obs_arr = next_obs_arr.copy()
            for lane in np.flatnonzero(dones):
                final_obs = infos[int(lane)].get("final_obs")
                if final_obs is not None:
                    transition_next_obs_arr[int(lane)] = obs_to_array(final_obs, self._obs_space)
            for lane in range(n_envs):
                self.buffer.add(
                    obs_arr[lane], action_flats[lane], float(rewards[lane]), transition_next_obs_arr[lane],
                    policy_targets[lane], bool(dones[lane]), lane=lane,
                    episode_outcome=terminal_self_play_outcome(
                        infos[lane],
                        bool(terminated[lane]),
                        bool(truncated[lane]),
                    ),
                )
            obs_arr = next_obs_arr
            ep_reward += rewards
            ep_length += 1
            prev_num_timesteps = num_timesteps
            num_timesteps += n_envs

            if self.buffer.num_episodes >= 1 and num_timesteps >= self.learning_starts:
                # `num_timesteps` jumps by `n_envs` each iteration (all lanes
                # step in lockstep), so a naive "did we cross a train_freq
                # boundary?" check fires at most once per iteration no matter
                # how many boundaries were actually crossed in that jump.
                # With num_envs > train_freq this silently divides the
                # gradient-steps-per-env-step ratio by n_envs, i.e. raising
                # num_envs makes training *less* sample-efficient purely as
                # a side effect of vectorization, not by design. Count the
                # boundaries actually crossed and run `train_steps_per_iter`
                # updates for each one, so the update-to-data ratio stays
                # constant regardless of num_envs.
                effective_prev = max(prev_num_timesteps, self.learning_starts)
                boundaries_crossed = (
                    num_timesteps // self.train_freq - effective_prev // self.train_freq
                )
                for _ in range(self.train_steps_per_iter * boundaries_crossed):
                    self._last_metrics = self._train_step()

                reanalyze_boundaries = (
                    num_timesteps // self.reanalyze_freq - effective_prev // self.reanalyze_freq
                )
                if reanalyze_boundaries > 0:
                    reanalyze_metrics = self._reanalyze()
                    if reanalyze_metrics:
                        self._last_metrics = {**self._last_metrics, **reanalyze_metrics}

            self._last_metrics = {
                **self._last_metrics,
                "replay_capacity_episodes": float(self.buffer.capacity),
                "replay_capacity_requested": float(self.hyperparams.get("buffer_size", 2_000)),
                **self_play_outcome_metrics(self),
            }
            any_done = False
            for lane in range(n_envs):
                if not dones[lane]:
                    continue
                any_done = True
                keep_going = callback.on_step(
                    num_timesteps, float(ep_reward[lane]), int(ep_length[lane]), self._last_metrics,
                )
                ep_reward[lane], ep_length[lane] = 0.0, 0
                if not keep_going:
                    return
            if not any_done:
                keep_going = callback.on_step(num_timesteps, metrics=self._last_metrics)
                if not keep_going:
                    return

    def predict(
        self,
        obs: Any,
        deterministic: bool = True,
        action_mask: np.ndarray | None = None,
    ) -> tuple[Any, Any]:
        obs_arr = obs_to_array(obs, self._obs_space)
        mask = (
            action_mask_from_info({"action_mask": action_mask}, int(self.n_actions))
            if self.discrete and action_mask is not None
            else mask_from_unwrapped_env(self.env, int(self.n_actions)) if self.discrete else None
        )
        action_masks = None if mask is None else mask[None]
        results = self.search(obs_arr[None], deterministic=np.array([deterministic]), action_masks=action_masks)
        return action_to_env(results[0]["env_action"], self._action_space), None

    def save(self, path: Path) -> None:
        torch.save(
            {
                "representation_state_dict": self.representation.state_dict(),
                "dynamics_state_dict": self.dynamics.state_dict(),
                "prediction_state_dict": self.prediction.state_dict(),
                "projector_state_dict": self.projector.state_dict(),
                "predictor_state_dict": self.predictor.state_dict(),
                "hyperparams": self.hyperparams,
            },
            path,
        )

    @classmethod
    def load(cls, path: Path, env: gym.Env, device: str = "cpu") -> "EfficientZeroV2":
        payload = torch.load(path, map_location="cpu", weights_only=False)
        algo = cls(env, payload.get("hyperparams", {}), None, device)
        algo.representation.load_state_dict(payload["representation_state_dict"])
        algo.dynamics.load_state_dict(payload["dynamics_state_dict"])
        algo.prediction.load_state_dict(payload["prediction_state_dict"])
        algo.projector.load_state_dict(payload["projector_state_dict"])
        algo.predictor.load_state_dict(payload["predictor_state_dict"])
        return algo



## Обучение

Случайные легальные ходы идут до `learning_starts`, дальше каждый ход выбирает Gumbel search. Градиентные шаги начинаются после первой законченной партии в буфере.

`PAPER_DEFAULTS = False` ставит 8 симуляций, маленький буфер и лимит партии 80 полуходов, чтобы градиенты начались сразу. Исходный рецепт — `PAPER_DEFAULTS = True`: буфер на 2000 партий, 32 симуляции, старт с 500 шагов и лимит 512 полуходов. Поднимите `TOTAL_TIMESTEPS`, когда нужен длинный прогон.


In [ ]:
"""Обучение.

`PAPER_DEFAULTS = False`: 8 симуляций, партии по 80 полуходов, 400 шагов.
`PAPER_DEFAULTS = True`: рецепт исходного прогона (32 симуляции, буфер 2000 партий,
старт с 500 шагов, лимит 512 полуходов, 2000 шагов).
`EZ_TIMESTEPS` в окружении переопределяет длину прогона. `NUM_ENVS > 1` собирает
несколько партий в одном процессе.
"""
PAPER_DEFAULTS = False
NUM_ENVS = 1

hyperparams = dict(DEFAULT_HYPERPARAMS)
if PAPER_DEFAULTS:
    # Лимит партии 512 полуходов задан в make_chess_env. 2000 шагов — это несколько партий.
    TOTAL_TIMESTEPS = int(os.environ.get("EZ_TIMESTEPS", "2000"))
else:
    # Короче 512, иначе первый градиент ждёт конца очень длинной партии.
    MAX_EPISODE_STEPS = 80
    TOTAL_TIMESTEPS = int(os.environ.get("EZ_TIMESTEPS", "400"))
    hyperparams.update(
        buffer_size=24,
        batch_size=16,
        learning_starts=40,
        num_simulations=8,
        num_top_actions=8,
        train_freq=4,
        train_steps_per_iter=1,
        reanalyze_batch_size=8,
    )


def build_train_env(num_envs: int) -> gym.Env:
    num_envs = max(1, int(num_envs))
    if num_envs == 1:
        return make_chess_env()
    from gymnasium.vector import AutoresetMode, SyncVectorEnv

    return SyncVectorEnv(
        [make_chess_env] * num_envs,
        autoreset_mode=AutoresetMode.SAME_STEP,
    )


class Progress:
    def __init__(self) -> None:
        self.episodes: list[tuple[int, float, int]] = []

    def on_step(
        self,
        num_timesteps: int,
        episode_reward: float | None = None,
        episode_length: float | None = None,
        metrics: dict | None = None,
    ) -> bool:
        metrics = metrics or {}
        loss = metrics.get("total_loss")
        loss_s = f"{loss:.4f}" if isinstance(loss, float) else "—"
        if episode_reward is not None:
            self.episodes.append((num_timesteps, float(episode_reward), int(episode_length or 0)))
            print(
                f"партий={len(self.episodes):3d}  шаг={num_timesteps:5d}  "
                f"длина={int(episode_length or 0):3d}  return={episode_reward:+.3f}  "
                f"loss={loss_s}  "
                f"решающие={metrics.get('self_play_decisive_rate', 0.0):.2f}  "
                f"ничьи={metrics.get('self_play_draw_rate', 0.0):.2f}"
            )
        elif num_timesteps % 20 == 0:
            print(
                f"шаг={num_timesteps:5d}  loss={loss_s}  "
                f"партий={metrics.get('self_play_games', 0):.0f}"
            )
        return True


train_env = build_train_env(NUM_ENVS)
agent = EfficientZeroV2(train_env, hyperparams, SEED, DEVICE)
progress = Progress()
agent.learn(TOTAL_TIMESTEPS, progress)
print(
    f"готово: шагов={TOTAL_TIMESTEPS}  партий={len(progress.episodes)}  "
    f"эпизодов в буфере={agent.buffer.num_episodes}"
)


## Проверка

Белые ходят поиском обученной сети, чёрные — случайным легальным ходом. Сохраняются веса `efficientzero_v2_chess.pt` в текущей директории.


In [ ]:
"""Партия обученной сети против случайного легального соперника.

Сеть играет белыми. Чёрные выбирают равномерно среди легальных ходов.
`agent.predict` запускает тот же Gumbel search, что и обучение.
"""
def play_vs_random(agent: EfficientZeroV2, seed: int = 1) -> dict:
    env = make_chess_env(render_mode="rgb_array")
    rng = np.random.default_rng(seed)
    obs, info = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0
    while True:
        white_to_move = int(info["current_player"]) == 1
        if white_to_move:
            action, _ = agent.predict(obs, deterministic=True, action_mask=info["action_mask"])
        else:
            legal = np.flatnonzero(info["action_mask"])
            action = int(rng.choice(legal)) if legal.size else 0
        obs, reward, terminated, truncated, info = env.step(int(action))
        total_reward += float(reward) if white_to_move else -float(reward)
        frame = env.render()
        if frame is not None:
            frames.append(frame)
        if terminated or truncated:
            final = info
            break
    board = env.unwrapped.game.render_ascii()
    env.close()
    winner = final.get("winner")
    outcome = {1: "победа белых", -1: "победа чёрных", 0: "ничья"}.get(winner, "лимит ходов")
    print(board)
    print(f"исход: {outcome}  полуходов={len(frames) - 1}  return белых={total_reward:+.3f}")
    return {"frames": frames, "winner": winner, "white_return": total_reward}


game = play_vs_random(agent, seed=SEED + 1)
try:
    from IPython.display import display
    from PIL import Image

    sheet = game["frames"][:: max(1, len(game["frames"]) // 6)]
    if sheet:
        display(Image.fromarray(sheet[-1]))
except Exception as exc:  # noqa: BLE001
    print("кадр доски не показан:", exc)

ckpt = Path("efficientzero_v2_chess.pt")
agent.save(ckpt)
print("веса сохранены в", ckpt.resolve())
